In [ ]:
# prompt: kết nối ggdrive

from google.colab import drive
drive.mount('/content/drive')
!chmod +x /content/drive/MyDrive/cplex/cplex

Mounted at /content/drive


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# prompt: tạo code test cho tôi cái cplex ở đường dẫn này: /content/drive/MyDrive/cplex/cplex

import subprocess

try:
  # Replace with the actual path to your CPLEX executable
  cplex_path = "/content/drive/MyDrive/cplex/cplex"

  # Run a simple command to check if CPLEX is accessible and executable
  # For example, you can try to get the version
  process = subprocess.run([cplex_path, "-v"], capture_output=True, text=True, check=True)

  print("CPLEX test successful. Output:")
  print(process.stdout)

except FileNotFoundError:
  print(f"Error: CPLEX executable not found at {cplex_path}")
except Exception as e:
  print(f"An error occurred while testing CPLEX: {e}")

CPLEX test successful. Output:



In [ ]:
!pip install PuLP

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.4/16.4 MB 63.7 MB/s eta 0:00:00


In [ ]:
# Enhanced E-VRPTW Model - Redesigned according to mathematical formulation

import math
import re
import os
import time
from typing import Dict, List, Tuple, Optional
import pulp as pl
import numpy as np
from datetime import datetime

class EVRPTWInstance:
    """Class to represent an E-VRPTW instance following the mathematical model"""

    def __init__(self):
        self.nodes = {}  # node_id -> {'type', 'x', 'y', 'demand', 'ready_time', 'due_date', 'service_time'}
        self.depot_id = None
        self.customers = []  # Set I
        self.stations = []   # Set F

        # Vehicle and battery parameters
        self.C = 0.0  # Vehicle capacity
        self.Q = 0.0  # Battery capacity
        self.r = 0.0  # Energy consumption rate per unit distance
        self.w = 0.0  # Wireless charging rate
        self.v = 0.0  # Average velocity

        # Wireless charging coverage
        self.omega = {}  # omega_ij: fraction of arc (i,j) with wireless charging [0,1]

        # Charging model parameters
        self.breakpoints = {}  # B = {0, 1, ..., b}
        self.num_breakpoints = 2

    def parse_instance_file(self, filename: str):
        """Parse the instance file format"""
        with open(filename, 'r') as f:
            lines = f.readlines()

        # Parse nodes
        parsing_nodes = True
        for line in lines:
            line = line.strip()
            if not line:
                continue

            if line.startswith('Q '):
                parsing_nodes = False
                match = re.search(r'/([0-9.]+)/', line)
                if match:
                    self.Q = float(match.group(1))
                    # self.Q = 1000
            elif line.startswith('C '):
                match = re.search(r'/([0-9.]+)/', line)
                if match:
                    self.C = float(match.group(1))
            elif line.startswith('r '):
                match = re.search(r'/([0-9.]+)/', line)
                if match:
                    self.r = float(match.group(1))
            elif line.startswith('g '):
                match = re.search(r'/([0-9.]+)/', line)
                if match:
                    # Use as base charging rate
                    self.g = float(match.group(1))
            elif line.startswith('v '):
                match = re.search(r'/([0-9.]+)/', line)
                if match:
                    self.v = float(match.group(1))
            elif line.startswith('w_charge '):
                match = re.search(r'/([0-9.]+)/', line)
                if match:
                    self.w = float(match.group(1))
            elif parsing_nodes and not line.startswith('StringID'):
                # Parse node line
                parts = line.split()
                if len(parts) >= 8:
                    node_id = parts[0]
                    node_type = parts[1]
                    x = float(parts[2])
                    y = float(parts[3])
                    demand = float(parts[4])
                    ready_time = float(parts[5])
                    due_date = float(parts[6])
                    service_time = float(parts[7])

                    self.nodes[node_id] = {
                        'type': node_type,
                        'x': x,
                        'y': y,
                        'demand': demand,
                        'ready_time': ready_time,
                        'due_date': due_date,
                        'service_time': service_time
                    }

                    if node_type == 'd':  # depot
                        self.depot_id = node_id
                    elif node_type == 'c':  # customer
                        self.customers.append(node_id)
                    elif node_type == 'f':  # fuel station
                        self.stations.append(node_id)

    def set_wireless_coverage(self, coverage_pattern: str = "moderate"):
        """Set wireless charging coverage omega_ij"""
        all_nodes = [self.depot_id] + self.customers + self.stations

        for i in all_nodes:
            for j in all_nodes:
                if i != j:
                    if coverage_pattern == "none":
                        self.omega[i, j] = 0.0
                    elif coverage_pattern == "light":
                        self.omega[i, j] = 0.2
                    elif coverage_pattern == "moderate":
                        self.omega[i, j] = 0.4
                    elif coverage_pattern == "high":
                        self.omega[i, j] = 0.6
                    elif coverage_pattern == "full":
                        self.omega[i, j] = 1.0
                    else:
                        self.omega[i, j] = 0.4

    def setup_charging_breakpoints(self):
        """Setup charging breakpoints for all replicated stations"""
        self.breakpoints = list(range(self.num_breakpoints + 1))  # B = {0, 1, ..., b}

        # For each station, define charging characteristics
        self.a = {}  # a_ik: battery level at breakpoint k at station i
        self.c = {}  # c_ik: cumulative charging time to breakpoint k at station i
        self.rho = {}  # rho_ik: charging rate in segment [a_ik, a_i,k+1]

         # ...existing code...
        for station in self.stations:
            for customer_idx in range(len(self.customers)):  # Replicate |I| times
                station_rep = f"{station}_rep_{customer_idx}"
                for k in self.breakpoints:
                    # Linear interpolation for battery levels
                    self.a[station_rep, k] = (k / self.num_breakpoints) * self.Q
                    # Non-linear charging time (example: square root function)
                    if k == 0:
                        self.c[station_rep, k] = 0.0
                    else:
                        charge_amount = self.a[station_rep, k]
                        alpha = 2.0
                        beta = 0.1
                        self.c[station_rep, k] = alpha * math.sqrt(charge_amount) + beta * charge_amount
                # Only compute rho for k in 0..num_breakpoints-1
                for k in range(self.num_breakpoints):
                    delta_battery = self.a[station_rep, k+1] - self.a[station_rep, k]
                    delta_time = self.c[station_rep, k+1] - self.c[station_rep, k]
                    self.rho[station_rep, k] = delta_battery / delta_time if delta_time > 0 else 1.0


In [ ]:
import os
import re
import subprocess

class EnhancedEVRPTWSolver:
    """Enhanced EVRPTW solver following the complete mathematical model"""

    def __init__(self, instance, max_vehicles: int = None, cplex_path: str = None):
        self.instance = instance
        if max_vehicles is None:
            self.max_vehicles = min(len(instance.customers), 4)
        else:
            self.max_vehicles = max_vehicles
        self.model = None
        self.solution = None

        # Set CPLEX path
        if cplex_path is None:
            self.cplex_path = "/content/drive/MyDrive/cplex/cplex"
        else:
            self.cplex_path = cplex_path

        self.mip_gap = None # Initialize mip_gap attribute


        # Setup problem structure according to mathematical model
        self._setup_problem_structure()

    def _setup_problem_structure(self):
        """Setup problem structure according to mathematical model"""
        # Set I: Customers
        self.I = self.instance.customers

        # Set F: Original stations
        self.F = self.instance.stations

        # Set F_rep: Replicated stations for multiple visits
        self.F_rep = []
        for station in self.F:
            for i in range(len(self.I)):  # Replicate |I| times
                self.F_rep.append(f"{station}_rep_{i}")

        # Set D: Depot start
        self.D = f"{self.instance.depot_id}_start"

        # Set E: Depot end
        self.E = f"{self.instance.depot_id}_end"

        # Set V: All vertices
        self.V = self.I + self.F_rep + [self.D, self.E]

        # Set A: All feasible arcs
        self.A = []
        for i in self.V:
            for j in self.V:
                if i != j and self._is_feasible_arc(i, j):
                    self.A.append((i, j))

        # Setup charging breakpoints
        self.instance.setup_charging_breakpoints()
        self.B = self.instance.breakpoints

    def _is_feasible_arc(self, i: str, j: str) -> bool:
      """Check if arc (i,j) is feasible"""
      if i == j:
          return False
      if i == self.E:  # No outgoing arcs from end depot
          return False
      if j == self.D:  # No incoming arcs to start depot
          return False
      # New condition: No arcs between replicas of the same charging station
      if self._is_station(i) and self._is_station(j):
          orig_i = self.get_original_node_id(i)
          orig_j = self.get_original_node_id(j)
          if orig_i == orig_j:
              return False
      return True

    def _is_station(self, vertex_id: str) -> bool:
        """Check if a vertex represents a charging station (original or replicated)"""
        original_id = self.get_original_node_id(vertex_id)
        return original_id in self.instance.stations

    def get_node_data(self, vertex_id: str) -> Dict:
        """Get node data for a vertex"""
        if vertex_id == self.D or vertex_id == self.E:
            return self.instance.nodes[self.instance.depot_id]
        elif vertex_id in self.instance.nodes:
            return self.instance.nodes[vertex_id]
        else:
            original_station = vertex_id.split('_rep_')[0]
            return self.instance.nodes[original_station]

    def get_original_node_id(self, vertex_id: str) -> str:
        """Get the original node ID"""
        if vertex_id == self.D or vertex_id == self.E:
            return self.instance.depot_id
        else:
            return vertex_id.split('_rep_')[0]

    def calculate_distance(self, i: str, j: str) -> float:
        """Calculate distance d_ij"""
        node_i = self.get_node_data(i)
        node_j = self.get_node_data(j)
        return math.sqrt((node_i['x'] - node_j['x'])**2 + (node_i['y'] - node_j['y'])**2)

    def calculate_travel_time(self, i: str, j: str) -> float:
        """Calculate travel time t_ij"""
        return self.calculate_distance(i, j) / self.instance.v

    def get_wireless_coverage(self, i: str, j: str) -> float:
        """Get wireless coverage omega_ij"""
        orig_i = self.get_original_node_id(i)
        orig_j = self.get_original_node_id(j)
        return self.instance.omega.get((orig_i, orig_j), 0.0)

    def create_model(self):
        """Create the optimization model according to mathematical formulation"""
        self.model = pl.LpProblem("Enhanced-EVRPTW-DWC-NL", pl.LpMinimize)

        # Decision variables
        self._create_variables()

        # Objective function
        self._create_objective()

        # Constraints
        self._create_constraints()

    def _create_variables(self):
        """Create all decision variables"""
        # x_ij: Binary routing variables
        self.x = {}
        for i, j in self.A:
            self.x[i, j] = pl.LpVariable(f"x_{i}_{j}", cat='Binary')

        # tau_j: Arrival time at node j
        self.tau = {}
        for j in self.V:
            node_data = self.get_node_data(j)
            self.tau[j] = pl.LpVariable(
                f"tau_{j}",
                lowBound=node_data['ready_time'],
                upBound=node_data['due_date']
            )

        # u_j: Remaining load when leaving node j
        self.u = {}
        for j in self.V:
            self.u[j] = pl.LpVariable(f"u_{j}", lowBound=0, upBound=self.instance.C)

        # y^a_j, y^d_j: Battery level when arriving/departing node j
        self.y_a = {}
        self.y_d = {}
        for j in self.V:
            self.y_a[j] = pl.LpVariable(f"y_a_{j}", lowBound=0, upBound=self.instance.Q)
            self.y_d[j] = pl.LpVariable(f"y_d_{j}", lowBound=0, upBound=self.instance.Q)

        # z_i: Station activation
        self.z = {}
        for i in self.F_rep:
            self.z[i] = pl.LpVariable(f"z_{i}", cat='Binary')

        # Piecewise linear approximation variables
        self.alpha = {}     # α_ik for arrival charge level
        self.lambd = {}     # λ_ik for departure charge level
        self.w_a = {}       # w^a_ik for arrival segment selection
        self.w_d = {}       # w^d_ik for departure segment selection

        for i in self.F_rep:
            for k in self.B:
                self.alpha[i, k] = pl.LpVariable(f"alpha_{i}_{k}", lowBound=0)
                self.lambd[i, k] = pl.LpVariable(f"lambda_{i}_{k}", lowBound=0)

                # Binary segment selection variables (exclude k=0)
                if k != 0:
                    self.w_a[i, k] = pl.LpVariable(f"w_a_{i}_{k}", cat='Binary')
                    self.w_d[i, k] = pl.LpVariable(f"w_d_{i}_{k}", cat='Binary')

        # Delta_i: Total charging time at station i
        self.Delta = {}
        for i in self.F_rep:
            self.Delta[i] = pl.LpVariable(f"Delta_{i}", lowBound=0)

    def _create_objective(self):
        """Create objective function (1)"""
        M1 = 3.0  # M1 >> M2
        M2 = 1.0

        # Fleet size: sum of vehicles departing from depot
        num_vehicles = pl.lpSum([self.x[self.D, j] for j in self.V if (self.D, j) in self.A])

        # Travel time
        travel_time = pl.lpSum([
            self.calculate_travel_time(i, j) * self.x[i, j]
            for i, j in self.A
        ])

        # Service time
        service_time = pl.lpSum([
            self.get_node_data(i)['service_time'] * pl.lpSum([
                self.x[j, i] for j in self.V if (j, i) in self.A
            ])
            for i in self.I
        ])

        # Charging time
        charging_time = pl.lpSum([self.Delta[i] for i in self.F_rep])

        # Total operational time
        total_time = travel_time + service_time + charging_time

        # Objective: minimize fleet size (primary) + operational time (secondary)
        self.model += M1 * num_vehicles + M2 * total_time

    def _create_constraints(self):
        """Create all constraints according to mathematical model"""

        # Customer Service and Flow Conservation Constraints

        # Constraint (2): Each customer must be visited exactly once
        for j in self.I:
            self.model += (
                pl.lpSum([self.x[i, j] for i in self.V if (i, j) in self.A]) == 1,
                f"customer_visit_{j}"
            )

        # Constraint (3): Each replicated station can be visited at most once
        for j in self.F_rep:
            self.model += (
                pl.lpSum([self.x[i, j] for i in self.V if (i, j) in self.A]) <= 1,
                f"station_visit_{j}"
            )

        # Constraint (4): Flow conservation
        for j in [v for v in self.V if v != self.D and v != self.E]:
            incoming = pl.lpSum([self.x[i, j] for i in self.V if (i, j) in self.A])
            outgoing = pl.lpSum([self.x[j, i] for i in self.V if (j, i) in self.A])
            self.model += (incoming == outgoing, f"flow_conservation_{j}")

        # Constraint (5): Vehicle balance
        vehicles_out = pl.lpSum([self.x[self.D, j] for j in self.V if (self.D, j) in self.A])
        vehicles_in = pl.lpSum([self.x[i, self.E] for i in self.V if (i, self.E) in self.A])
        self.model += (vehicles_out == vehicles_in, "vehicle_balance")

        # Charging Station Visit and Activation Constraints

        # Constraint (6): Station activation
        for i in self.F_rep:
            self.model += (
                self.z[i] == pl.lpSum([self.x[j, i] for j in self.V if (j, i) in self.A]),
                f"station_activation_{i}"
            )

        # Temporal Feasibility Constraints

        # Constraint (7): Depot start time
        self.model += (self.tau[self.D] == 0, "depot_start_time")

        # Constraint (8): Time feasibility for customers and depot
        big_M = max([self.get_node_data(v)['due_date'] for v in self.V]) * 2
        for i in self.I + [self.D]:
            for j in self.V:
                if (i, j) in self.A:
                    node_i = self.get_node_data(i)
                    travel_time = self.calculate_travel_time(i, j)
                    service_time = node_i['service_time']

                    self.model += (
                        self.tau[i] + travel_time + service_time <=
                        self.tau[j] + big_M * (1 - self.x[i, j]),
                        f"time_feasibility_customer_{i}_{j}"
                    )

        # Constraint (9): Time feasibility for stations
        for i in self.F_rep:
            for j in self.V:
                if (i, j) in self.A:
                    travel_time = self.calculate_travel_time(i, j)

                    self.model += (
                        self.tau[i] + travel_time + self.Delta[i] <=
                        self.tau[j] + big_M * (1 - self.x[i, j]),
                        f"time_feasibility_station_{i}_{j}"
                    )

        # Constraint (10): Time windows
        for j in self.V:
            node_data = self.get_node_data(j)
            self.model += (
                self.tau[j] >= node_data['ready_time'],
                f"time_window_early_{j}"
            )
            self.model += (
                self.tau[j] <= node_data['due_date'],
                f"time_window_late_{j}"
            )

        # Capacity Constraints

        # Constraint (11): Load feasibility
        for i in self.V:
            for j in self.I:
                if (i, j) in self.A:
                    node_j = self.get_node_data(j)
                    self.model += (
                        self.u[j] <= self.u[i] - node_j['demand'] * self.x[i, j] +
                        self.instance.C * (1 - self.x[i, j]),
                        f"load_feasibility_{i}_{j}"
                    )

        # Constraint (12): Initial load
        self.model += (self.u[self.D] == self.instance.C, "initial_load")

        # Enhanced Energy Management Constraints with Wireless Charging

        # Constraints (13) and (14): Battery management with wireless charging
        for i, j in self.A:
            distance = self.calculate_distance(i, j)
            wireless_coverage = self.get_wireless_coverage(i, j)
            consumption = self.instance.r * distance
            wireless_charge = self.instance.w * distance * wireless_coverage
            big_M_energy = self.instance.Q * 2 + wireless_charge

            # Constraint (13): Upper bound
            self.model += (
                self.y_a[j] <= self.y_d[i] - consumption + wireless_charge + big_M_energy * (1 - self.x[i, j]),
                f"battery_management_upper_{i}_{j}"
            )

            # Constraint (14): Lower bound
            self.model += (
                self.y_a[j] >= self.y_d[i] - consumption + wireless_charge - big_M_energy * (1 - self.x[i, j]),
                f"battery_management_lower_{i}_{j}"
            )

        # Constraint (15): Battery continuity for non-stations
        for j in self.I + [self.D, self.E]:
            self.model += (self.y_d[j] == self.y_a[j], f"battery_continuity_{j}")

        # Constraint (16): Initial battery level
        self.model += (self.y_d[self.D] == self.instance.Q, "initial_battery")

        # Piecewise Linear Charging Function Constraints for Arrival States

        for i in self.F_rep:
            # Constraint (17): Arrival charge level interpolation
            self.model += (
                self.y_a[i] == pl.lpSum([self.alpha[i, k] * self.instance.a.get((i, k), 0) for k in self.B]),
                f"arrival_interpolation_{i}"
            )

            # Constraint (18): Normalization of arrival coefficients
            self.model += (
                pl.lpSum([self.alpha[i, k] for k in self.B]) == self.z[i],
                f"arrival_normalization_{i}"
            )

            # Constraint (19): Arrival segment selection limit
            self.model += (
                pl.lpSum([self.w_a[i, k] for k in self.B if k != 0]) == self.z[i],
                f"arrival_segment_limit_{i}"
            )

            # Constraint (20): First breakpoint bound
            if len(self.B) > 1:
                self.model += (
                    self.alpha[i, 0] <= self.w_a[i, self.B[1]],
                    f"arrival_first_bound_{i}"
                )

            # Constraint (21): Middle breakpoint bounds
            for k_idx in range(1, len(self.B) - 1):
                k = self.B[k_idx]
                k_next = self.B[k_idx + 1]
                self.model += (
                    self.alpha[i, k] <= self.w_a[i, k] + self.w_a[i, k_next],
                    f"arrival_middle_bound_{i}_{k}"
                )

            # Constraint (22): Last breakpoint bound
            if len(self.B) > 1:
                k_last = self.B[-1]
                self.model += (
                    self.alpha[i, k_last] <= self.w_a[i, k_last],
                    f"arrival_last_bound_{i}"
                )

            # Constraint (23): Non-adjacent segment restriction
            for k_idx in range(len(self.B) - 2):
                k = self.B[k_idx]
                k_plus_2 = self.B[k_idx + 2]
                if k != 0 and k_plus_2 != 0:
                    self.model += (
                        self.w_a[i, k] + self.w_a[i, k_plus_2] <= 1,
                        f"arrival_non_adjacent_{i}_{k}_{k_plus_2}"
                    )

        # Piecewise Linear Charging Function Constraints for Departure States

        for i in self.F_rep:
            # Constraint (24): Departure charge level interpolation
            self.model += (
                self.y_d[i] == pl.lpSum([self.lambd[i, k] * self.instance.a.get((i, k), 0) for k in self.B]),
                f"departure_interpolation_{i}"
            )

            # Constraint (25): Normalization of departure coefficients
            self.model += (
                pl.lpSum([self.lambd[i, k] for k in self.B]) == self.z[i],
                f"departure_normalization_{i}"
            )

            # Constraint (26): Departure segment selection limit
            self.model += (
                pl.lpSum([self.w_d[i, k] for k in self.B if k != 0]) == self.z[i],
                f"departure_segment_limit_{i}"
            )

            # Constraint (27): First breakpoint bound
            if len(self.B) > 1:
                self.model += (
                    self.lambd[i, 0] <= self.w_d[i, self.B[1]],
                    f"departure_first_bound_{i}"
                )

            # Constraint (28): Middle breakpoint bounds
            for k_idx in range(1, len(self.B) - 1):
                k = self.B[k_idx]
                k_next = self.B[k_idx + 1]
                self.model += (
                    self.lambd[i, k] <= self.w_d[i, k] + self.w_d[i, k_next],
                    f"departure_middle_bound_{i}_{k}"
                )

            # Constraint (29): Last breakpoint bound
            if len(self.B) > 1:
                k_last = self.B[-1]
                self.model += (
                    self.lambd[i, k_last] <= self.w_d[i, k_last],
                    f"departure_last_bound_{i}"
                )

            # Constraint (30): Non-adjacent segment restriction
            for k_idx in range(len(self.B) - 2):
                k = self.B[k_idx]
                k_plus_2 = self.B[k_idx + 2]
                if k != 0 and k_plus_2 != 0:
                    self.model += (
                        self.w_d[i, k] + self.w_d[i, k_plus_2] <= 1,
                        f"departure_non_adjacent_{i}_{k}_{k_plus_2}"
                    )

        # Nonlinear Charging Time Calculation

        # Constraint (31): Charging time calculation
        for i in self.F_rep:
            departure_time = pl.lpSum([self.lambd[i, k] * self.instance.c.get((i, k), 0) for k in self.B])
            arrival_time = pl.lpSum([self.alpha[i, k] * self.instance.c.get((i, k), 0) for k in self.B])

            self.model += (
                self.Delta[i] == departure_time - arrival_time,
                f"charging_time_{i}"
            )

        # Partial Charging and Battery State Consistency Constraints

        # Constraint (32): Battery non-decreasing during charging
        for i in self.F_rep:
            self.model += (self.y_a[i] <= self.y_d[i], f"battery_non_decreasing_{i}")

        # Additional constraint: Maximum vehicles
        vehicles_out = pl.lpSum([self.x[self.D, j] for j in self.V if (self.D, j) in self.A])
        self.model += (vehicles_out <= self.max_vehicles, "max_vehicles_constraint")

    # def solve(self, time_limit: int = 3600, log_file: str = None) -> bool:
    #     """Solve the model"""
    #     if self.model is None:
    #         self.create_model()

    #     solver_name = "CBC"
    #     solver = None
    #     if os.path.exists(self.cplex_path):
    #         try:
    #             solver = pl.CPLEX_CMD(
    #                 path=self.cplex_path,
    #                 msg=1,  # Keep solver messages on to capture output
    #                 timeLimit=time_limit,
    #                 gapRel=0.001,
    #                 logPath=log_file if log_file else None # Direct log to file if provided
    #             )
    #             solver_name = "CPLEX"
    #         except Exception as e:
    #              print(f"Warning: Could not initialize CPLEX_CMD: {e}. Falling back to CBC.")
    #              solver = pl.PULP_CBC_CMD(msg=0, timeLimit=time_limit, gapRel=0.001)
    #              solver_name = "CBC"
    #     else:
    #         solver = pl.PULP_CBC_CMD(msg=0, timeLimit=time_limit, gapRel=0.001)


    #     print(f"Using solver: {solver_name}")

    #     start_time = time.time()
    #     self.model.solve(solver)
    #     self.solve_time = time.time() - start_time
    #     self.solver_used = solver_name

    #     self.mip_gap = None # Reset mip_gap

    #     # Try to get the MIP gap after solving if CPLEX was used
    #     if solver_name == "CPLEX":
    #          if self.mip_gap is None and log_file and os.path.exists(log_file):
    #              try:
    #                  with open(log_file, 'r') as f:
    #                      log_content = f.read()
    #                  # This regex looks for the last occurrence of "Current MIP" and captures the gap percentage
    #                  match = re.search(r"Current MIP\s+.*?Gap = ([\d\.]+) %", log_content, re.DOTALL)
    #                  if match:
    #                      self.mip_gap = float(match.group(1)) / 100.0 # Convert percentage to fraction
    #                      print(f"MILP_GAP parsed from log: {self.mip_gap:.4f}")
    #                  else:
    #                      # print("Warning: Could not parse MILP gap from CPLEX log.") # Keep for debugging
    #                      pass
    #              except Exception as e:
    #                  print(f"Error reading or parsing log file {log_file}: {e}")
    #                  self.mip_gap = None
    #       # Check the model status after solving
    #     if self.model.status in (pl.LpStatusOptimal, pl.LpStatusNotOptimal, pl.LpStatusFeasible):
    #         print("Solver found a feasible solution.")
    #         # You can then proceed with extracting the solution if needed
    #         self._extract_solution() # Call extract_solution here
    #         return True # Or just continue if you don't need to return True/False
    #     elif self.model.status == pl.LpStatusInfeasible:
    #         print("Solver found the model is infeasible.")
    #         self.solution = None # Ensure solution is None
    #         return False
    #     else:
    #         print(f"Solver status: {pl.LpStatus[self.model.status]} - No feasible solution found.")
    #         self.solution = None # Ensure solution is None
    #         return False
    def solve(self, time_limit: int = 3600, log_file: str = None) -> bool:
        """Solve the model"""
        if self.model is None:
            self.create_model()

        if os.path.exists(self.cplex_path):
            try:
                solver = pl.CPLEX_CMD(
                    path=self.cplex_path,
                    msg=1,  # Keep solver messages on to capture output
                    timeLimit=time_limit,
                    gapRel=0.001,
                    logPath=log_file if log_file else None # Direct log to file if provided
                )
                solver_name = "CPLEX"
            except Exception as e:
                 print(f"Warning: Could not initialize CPLEX_CMD: {e}. Falling back to CBC.")
                 solver = pl.PULP_CBC_CMD(msg=0, timeLimit=time_limit, gapRel=0.001)
                 solver_name = "CBC"
        else:
            solver = pl.PULP_CBC_CMD(msg=0, timeLimit=time_limit, gapRel=0.001)

        print(f"Using solver: {solver_name}")

        start_time = time.time()
        self.model.solve(solver)
        self.solve_time = time.time() - start_time
        self.solver_used = solver_name
        # if solver_name == "CPLEX":
        #      # Access the solver's solution object to get the gap
        #      if self.model.status == pl.LpStatusOptimal or self.model.status == pl.LpStatusNotOptimal:
        #          try:
        #              # Use the model's solution object to get the gap
        #              self.mip_gap = self.model.solution.get_MIP_relative_gap()
        #          except Exception as e:
        #              print(f"Warning: Could not retrieve MILP gap from CPLEX solution: {e}")
        #              self.mip_gap = None
            # Remove the line that caused the error
            # if solver_name == "CPLEX":
            #     self.mip_gap = solver.get_mip_relative_gap()
            # else:
            #     self.mip_gap = None # Or get gap for CBC if available


        if self.model.status == pl.LpStatusOptimal:
            self._extract_solution()
            return True
        else:
            return False


    def _extract_solution(self):
        """Extract solution from solved model, focusing on objective-related variables"""
        # Check if model status indicates a solution exists before accessing values

        # Ensure variables have values before accessing them
        try:
            num_vehicles = sum(pl.value(self.x[i, j]) for i, j in self.A if i == self.D)
            travel_time = sum(
                self.calculate_travel_time(i, j) * pl.value(self.x[i, j])
                for i, j in self.A
            )
            service_time = sum(
                self.get_node_data(i)['service_time'] * sum(
                    pl.value(self.x[j, i]) for j in self.V if (j, i) in self.A
                )
                for i in self.I
            )
            charging_time = sum(
                pl.value(self.Delta[i]) for i in self.F_rep if pl.value(self.z[i]) > 0.5
            )
            total_time = travel_time + service_time + charging_time
            total_cost = pl.value(self.model.objective) # Prefer objective value if available
            if total_cost is None:
                 total_cost = 10000.0 * num_vehicles + 1.0 * total_time # Fallback calculation

        except Exception as e:
            print(f"Warning: Could not extract detailed solution components: {e}")
            self.solution = None
            return


        self.solution = {
            'num_vehicles': num_vehicles,
            'total_time': total_time,
            'total_cost': total_cost,
            'routes': self._extract_routes(),
            'charging_details': self._extract_charging_details(),
            'route_battery_levels': self._extract_route_battery_levels(),
            'computation_time': self.solve_time,
            'solver_used': self.solver_used,
            'mip_gap': self.mip_gap # Include mip_gap in the solution
        }

    def print_solution(self):
        """Print solution summary and return as dict for further processing"""
        if not self.solution:
            print("No solution found!")
            return None

        print(f"=== Solution Summary ===")
        print(f"Number of vehicles: {self.solution.get('num_vehicles', '-')}")
        print(f"Total time: {self.solution.get('total_time', '-')}")
        print(f"Total cost: {self.solution.get('total_cost', '-')}")
        print(f"Computation time: {self.solution.get('computation_time', '-'):.2f}s" if isinstance(self.solution.get('computation_time'), (int, float)) else f"Computation time: {self.solution.get('computation_time', '-')}")
        print(f"Solver used: {self.solution.get('solver_used', '-')}")
        if self.solution.get('mip_gap') is not None:
            print(f"MILP GAP: {self.solution['mip_gap']:.4f}")


        print(f"\n=== Routes & Battery Levels ===")
        if self.solution.get('routes'):
            for i, route in enumerate(self.solution['routes']):
                print(f"Vehicle {i+1}: {' -> '.join(route)}")
                if self.solution.get('route_battery_levels') and i < len(self.solution['route_battery_levels']):
                     for node, arrival_level, departure_level in self.solution['route_battery_levels'][i]:
                         if departure_level is not None:
                             print(f"    {node}: Arrival={arrival_level:.2f}, Departure={departure_level:.2f}")
                         else:
                             print(f"    {node}: Battery={arrival_level:.2f}")
        else:
            print("No routes found in solution.")


        # if self.solution['charging_details']:
        #   print(f"\n=== Charging Details ===")
        #   for station_id, visits in self.solution['charging_details'].items():
        #       print(f"Station {station_id}:")
        #       for idx, details in enumerate(visits, 1):
        #           print(f"  Visit {idx}: Time={details['charging_time']:.2f}, "
        #                 f"Battery: {details['battery_before']:.2f} -> {details['battery_after']:.2f}")


            # for original_station, visits in station_groups.items():
            #     print(f"Station {original_station}:")
            #     for idx, details in enumerate(visits, 1):
            #         print(f"  Visit {idx}: Time={details['charging_time']:.2f}, "
            #               f"Battery: {details['battery_before']:.2f} -> {details['battery_after']:.2f}")

        return {
            'vehicles': self.solution.get('num_vehicles', '-'),
            'time': self.solution.get('total_time', '-'),
            'total_cost': self.solution.get('total_cost', '-'),
            'routes': self.solution.get('routes', []),
            'charging_details': self.solution.get('charging_details', {}),
            'route_battery_levels': self.solution.get('route_battery_levels', []),
            'mip_gap': self.solution.get('mip_gap') # Include mip_gap in the returned dict
        }

    def _extract_routes(self) -> List[List[str]]:
        """Extract route information, handling replicated charging stations"""
        routes = []

        used_arcs = [(i, j) for i, j in self.A if pl.value(self.x[i, j]) > 0.5]
        depot_arcs = [(i, j) for i, j in used_arcs if i == self.D]
        visited = set()

        for depot_arc in depot_arcs:
            route = [self.instance.depot_id]
            current = depot_arc[1]
            current_arc = depot_arc
            route_arcs = [current_arc]

            while current != self.E and current not in visited:
                visited.add(current)
                node_label = self.get_original_node_id(current)
                if node_label != self.instance.depot_id or current == self.E:
                    route.append(node_label)

                next_arcs = [(i, j) for i, j in used_arcs if i == current and j not in visited]
                if not next_arcs:
                    if current != self.E:
                        route.append(self.instance.depot_id)
                    break

                current_arc = next_arcs[0]
                route_arcs.append(current_arc)
                current = current_arc[1]

            if current == self.E and self.instance.depot_id not in route[-1:]:
                route.append(self.instance.depot_id)

            if len(route) > 2:  # Ensure valid route (depot -> at least one node -> depot)
                routes.append(route)

        return routes

    def _extract_route_battery_levels(self) -> List[List[Tuple[str, float, float]]]:
        """Extract battery levels at each node in routes, including arrival and departure levels"""
        battery_levels = []

        used_arcs = [(i, j) for i, j in self.A if pl.value(self.x[i, j]) > 0.5]
        depot_arcs = [(i, j) for i, j in used_arcs if i == self.D]
        visited = set()

        for depot_arc in depot_arcs:
            route_battery = []
            # Add depot start battery level, only if value is not None
            y_a_D_value = pl.value(self.y_a[self.D])
            y_d_D_value = pl.value(self.y_d[self.D])
            if y_a_D_value is not None and y_d_D_value is not None:
                route_battery.append((self.instance.depot_id, y_a_D_value, y_d_D_value))
            else:
                 route_battery.append((self.instance.depot_id, None, None))


            current = depot_arc[1]
            visited.add(self.D)

            while current != self.E and current not in visited:
                visited.add(current)
                node_label = self.get_original_node_id(current)
                # Add node battery levels, only if values are not None
                y_a_current_value = pl.value(self.y_a[current])
                y_d_current_value = pl.value(self.y_d[current])

                departure_level = y_d_current_value if current in self.F_rep else None

                if y_a_current_value is not None:
                    route_battery.append((node_label, y_a_current_value, departure_level))
                else:
                     route_battery.append((node_label, None, departure_level))


                next_arcs = [(i, j) for i, j in used_arcs if i == current and j not in visited]
                if not next_arcs:
                    if current != self.E:
                        # Add depot end battery level, only if value is not None
                        y_a_E_value = pl.value(self.y_a[self.E])
                        if y_a_E_value is not None:
                            route_battery.append((self.instance.depot_id, y_a_E_value, pl.value(self.y_d[self.E])))
                        else:
                            route_battery.append((self.instance.depot_id, None, None))
                    break

                current = next_arcs[0][1]

            if current == self.E and route_battery[-1][0] != self.instance.depot_id:
                 y_a_E_value = pl.value(self.y_a[self.E])
                 if y_a_E_value is not None:
                     route_battery.append((self.instance.depot_id, y_a_E_value, pl.value(self.y_d[self.E])))
                 else:
                     route_battery.append((self.instance.depot_id, None, None))


            if len(route_battery) > 2:  # Ensure valid route
                battery_levels.append(route_battery)

        return battery_levels

    def _extract_charging_details(self) -> Dict[str, List[Dict]]:
      """Extract charging details grouped by original station ID"""
      charging_details = {}
      for i in self.F_rep:
          # Check if z[i], Delta[i], y_a[i], y_d[i] are not None before calling pl.value()
          if self.z.get(i) is not None and pl.value(self.z[i]) > 0.5 and \
             self.Delta.get(i) is not None and pl.value(self.Delta[i]) is not None and pl.value(self.Delta[i]) > 1e-4 and \
             self.y_a.get(i) is not None and pl.value(self.y_a[i]) is not None and \
             self.y_d.get(i) is not None and pl.value(self.y_d[i]) is not None:

              original_station = self.get_original_node_id(i)
              if original_station not in charging_details:
                  charging_details[original_station] = []

              charging_details[original_station].append({
                  'replicated_id': i,
                  'charging_time': pl.value(self.Delta[i]),
                  'battery_before': pl.value(self.y_a[i]),
                  'battery_after': pl.value(self.y_d[i]),
                  'charge_amount': pl.value(self.y_d[i]) - pl.value(self.y_a[i])
              })
      return charging_details



if __name__ == "__main__":
    print("Enhanced EVRPTW")

Enhanced EVRPTW


In [ ]:
import math
import time
from typing import Dict, List, Tuple
import pulp as pl
import os
import re

class EnhancedEVRPTW_NL:
    """Enhanced EVRPTW solver adjusted to match Montoya et al. (2017) MILP constraints with wireless charging"""

    def __init__(self, instance, max_vehicles: int = 100, cplex_path: str = None):
        self.instance = instance
        self.max_vehicles = max_vehicles
        self.cplex_path = cplex_path or "/content/drive/MyDrive/cplex/cplex"
        self.model = None
        self.solution = None
        self.mip_gap = None
        self._setup_problem_structure()

    def _setup_problem_structure(self):
        self.I = self.instance.customers
        self.F = self.instance.stations
        self.D = self.instance.depot_id
        self.E = self.instance.depot_id
        self.V = self.I + self.F + [self.D, self.E]
        self.F_rep = self.F
        self.A = []
        for i in self.V:
            for j in self.V:
                if self._is_feasible_arc(i, j):
                    self.A.append((i, j))
        for arc in self.A:
            if not isinstance(arc, tuple) or len(arc) != 2:
                raise ValueError(f"Invalid arc format: {arc}")
        self.instance.setup_charging_breakpoints()
        self.B = self.instance.breakpoints
        self.Tmax = getattr(self.instance, 'Tmax', 1440)  # Default Tmax if not provided
        self.Smax = max(self.instance.c.get((i, self.B[-1]), 0) for i in self.F_rep)  # Max charging time
        self.m = {(i, j): 1 if i in self.F_rep and j in self.F_rep and i == j else 0 for i in self.V for j in self.V}  # Symmetry parameter

    def _is_feasible_arc(self, i: str, j: str) -> bool:
        if i == j or i == self.E or j == self.D:
            return False
        if self._is_station(i) and self._is_station(j):
            orig_i = self.get_original_node_id(i)
            orig_j = self.get_original_node_id(j)
            if orig_i == orig_j:
                return False
        return True

    def _is_station(self, vertex_id: str) -> bool:
        return vertex_id in self.F_rep

    def get_original_node_id(self, vertex_id: str) -> str:
        return vertex_id

    def _get_node_data(self, vertex_id: str) -> Dict:
        return self.instance.nodes.get(vertex_id, {'x': 0, 'y': 0, 'demand': 0, 'ready_time': 0, 'due_date': math.inf, 'service_time': 0})

    def _calculate_distance(self, i: str, j: str) -> float:
        node_i = self._get_node_data(i)
        node_j = self._get_node_data(j)
        return math.sqrt((node_j['x'] - node_i['x']) ** 2 + (node_j['y'] - node_i['y']) ** 2)

    def _calculate_travel_time(self, i: str, j: str) -> float:
        return self._calculate_distance(i, j) / self.instance.v

    def _calculate_energy_consumption(self, i: str, j: str) -> float:
        return self.instance.r * self._calculate_distance(i, j)

    def _get_wireless_coverage(self, i: str, j: str) -> float:
        return self.instance.omega.get((i, j), 0)

    def create_model(self):
        self.model = pl.LpProblem("EVRPTW", pl.LpMinimize)
        self._create_variables()
        self._create_objective()
        self._create_constraints()

    def _create_variables(self):
        self.x = {(i, j): pl.LpVariable(f"x_{i}_{j}", cat='Binary') for i, j in self.A}
        self.tau = {j: pl.LpVariable(f"tau_{j}", lowBound=0) for j in self.V}  # Departure time
        self.u = {j: pl.LpVariable(f"u_{j}", lowBound=0, upBound=self.instance.C) for j in self.V}
        self.y_a = {j: pl.LpVariable(f"y_a_{j}", lowBound=0, upBound=self.instance.Q) for j in self.V}
        self.y_d = {j: pl.LpVariable(f"y_d_{j}", lowBound=0, upBound=self.instance.Q) for j in self.V}
        self.z = {i: pl.LpVariable(f"z_{i}", cat='Binary') for i in self.F_rep}
        self.alpha = {(i, k): pl.LpVariable(f"alpha_{i}_{k}", lowBound=0) for i in self.F_rep for k in self.B}
        self.lambd = {(i, k): pl.LpVariable(f"lambda_{i}_{k}", lowBound=0) for i in self.F_rep for k in self.B}
        self.w_a = {(i, k): pl.LpVariable(f"w_a_{i}_{k}", cat='Binary') for i in self.F_rep for k in self.B if k != 0}
        self.w_d = {(i, k): pl.LpVariable(f"w_d_{i}_{k}", cat='Binary') for i in self.F_rep for k in self.B if k != 0}
        self.Delta = {i: pl.LpVariable(f"Delta_{i}", lowBound=0) for i in self.F_rep}

    def _create_objective(self):
        M1 = 10000.0
        M2 = 1.0
        num_vehicles = pl.lpSum([self.x[self.D, j] for j in self.V if (self.D, j) in self.A])
        travel_time = pl.lpSum([self._calculate_travel_time(i, j) * self.x[i, j] for i, j in self.A])
        service_time = pl.lpSum([self._get_node_data(i)['service_time'] * pl.lpSum([self.x[j, i] for j in self.V if (j, i) in self.A]) for i in self.I])
        charging_time = pl.lpSum([self.Delta[i] for i in self.F_rep])
        total_time = travel_time + service_time + charging_time
        self.model += M1 * num_vehicles + M2 * total_time

    def _create_constraints(self):
        # Big-M values
        big_M = self.Tmax + self.Smax
        big_M_energy = self.instance.Q * 2

        # Constraint (2): Each customer visited once
        for j in self.I:
            self.model += (pl.lpSum([self.x[i, j] for i in self.V if (i, j) in self.A]) == 1, f"customer_visit_{j}")

        # Constraint (3): Each CS visited at most once
        for j in self.F_rep:
            self.model += (pl.lpSum([self.x[i, j] for i in self.V if (i, j) in self.A]) <= 1, f"station_visit_{j}")

        # Constraint (4): Flow conservation
        for j in [v for v in self.V if v != self.D and v != self.E]:
            incoming = pl.lpSum([self.x[i, j] for i in self.V if (i, j) in self.A])
            outgoing = pl.lpSum([self.x[j, i] for i in self.V if (j, i) in self.A])
            self.model += (incoming == outgoing, f"flow_conservation_{j}")

        # Constraint (5): Vehicle balance
        vehicles_out = pl.lpSum([self.x[self.D, j] for j in self.V if (self.D, j) in self.A])
        vehicles_in = pl.lpSum([self.x[i, self.E] for i in self.V if (i, self.E) in self.A])
        self.model += (vehicles_out == vehicles_in, "vehicle_balance")

        # Constraint (6): Station activation
        for i in self.F_rep:
            self.model += (self.z[i] == pl.lpSum([self.x[j, i] for j in self.V if (j, i) in self.A]), f"station_activation_{i}")

        # Constraints (5-6 in MILP): Battery tracking with wireless charging
        for i, j in self.A:
            distance = self._calculate_distance(i, j)
            consumption = self._calculate_energy_consumption(i, j)
            wireless_charge = self.instance.w * distance * self._get_wireless_coverage(i, j)
            if j in self.I:
                self.model += (self.y_d[i] - self.y_d[j] <= consumption - wireless_charge + big_M_energy * (1 - self.x[i, j]), f"battery_customer_upper_{i}_{j}")
                self.model += (self.y_d[i] - self.y_d[j] >= consumption - wireless_charge - big_M_energy * (1 - self.x[i, j]), f"battery_customer_lower_{i}_{j}")
            elif j in self.F_rep:
                self.model += (self.y_d[i] - self.y_a[j] <= consumption - wireless_charge + big_M_energy * (1 - self.x[i, j]), f"battery_station_upper_{i}_{j}")
                self.model += (self.y_d[i] - self.y_a[j] >= consumption - wireless_charge - big_M_energy * (1 - self.x[i, j]), f"battery_station_lower_{i}_{j}")

        # Constraint (7 in MILP): Sufficient energy to depot
        for i in self.V:
            if (i, self.E) in self.A:
                e_iE = self._calculate_energy_consumption(i, self.E)
                wireless_charge = self.instance.w * self._calculate_distance(i, self.E) * self._get_wireless_coverage(i, self.E)
                self.model += (self.y_d[i] >= (e_iE - wireless_charge) * self.x[i, self.E], f"energy_to_depot_{i}")

        # Constraint (8 in MILP): Battery reset at CS
        for i in self.F_rep:
            self.model += (self.y_d[i] == self.y_a[i], f"battery_reset_{i}")

        # Constraint (9 in MILP): Full battery at depot
        self.model += (self.y_d[self.D] == self.instance.Q, "initial_battery")

        # Constraint (10 in MILP): Battery non-decreasing at CS
        for i in self.F_rep:
            self.model += (self.y_a[i] <= self.y_d[i], f"battery_non_decreasing_{i}")

        # Constraints (11-17 in MILP): Arrival charge level at CS
        for i in self.F_rep:
            self.model += (self.y_a[i] == pl.lpSum([self.alpha[i, k] * self.instance.a.get((i, k), 0) for k in self.B]), f"arrival_interpolation_{i}")
            self.model += (pl.lpSum([self.alpha[i, k] for k in self.B]) == pl.lpSum([self.w_a[i, k] for k in self.B if k != 0]), f"arrival_normalization_{i}")
            self.model += (pl.lpSum([self.w_a[i, k] for k in self.B if k != 0]) == self.z[i], f"arrival_segment_limit_{i}")
            if len(self.B) > 1:
                self.model += (self.alpha[i, 0] <= self.w_a[i, self.B[1]], f"arrival_first_bound_{i}")
                self.model += (self.alpha[i, self.B[-1]] <= self.w_a[i, self.B[-1]], f"arrival_last_bound_{i}")
                for k_idx in range(1, len(self.B) - 1):
                    k = self.B[k_idx]
                    k_next = self.B[k_idx + 1]
                    self.model += (self.alpha[i, k] <= self.w_a[i, k] + self.w_a[i, k_next], f"arrival_middle_bound_{i}_{k}")
                for k_idx in range(len(self.B) - 2):
                    k = self.B[k_idx]
                    k_plus_2 = self.B[k_idx + 2]
                    if k != 0 and k_plus_2 != 0:
                        self.model += (self.w_a[i, k] + self.w_a[i, k_plus_2] <= 1, f"arrival_non_adjacent_{i}_{k}_{k_plus_2}")

        # Constraints (18-24 in MILP): Departure charge level at CS
        for i in self.F_rep:
            self.model += (self.y_d[i] == pl.lpSum([self.lambd[i, k] * self.instance.a.get((i, k), 0) for k in self.B]), f"departure_interpolation_{i}")
            self.model += (pl.lpSum([self.lambd[i, k] for k in self.B]) == pl.lpSum([self.w_d[i, k] for k in self.B if k != 0]), f"departure_normalization_{i}")
            self.model += (pl.lpSum([self.w_d[i, k] for k in self.B if k != 0]) == self.z[i], f"departure_segment_limit_{i}")
            if len(self.B) > 1:
                self.model += (self.lambd[i, 0] <= self.w_d[i, self.B[1]], f"departure_first_bound_{i}")
                self.model += (self.lambd[i, self.B[-1]] <= self.w_d[i, self.B[-1]], f"departure_last_bound_{i}")
                for k_idx in range(1, len(self.B) - 1):
                    k = self.B[k_idx]
                    k_next = self.B[k_idx + 1]
                    self.model += (self.lambd[i, k] <= self.w_d[i, k] + self.w_d[i, k_next], f"departure_middle_bound_{i}_{k}")
                for k_idx in range(len(self.B) - 2):
                    k = self.B[k_idx]
                    k_plus_2 = self.B[k_idx + 2]
                    if k != 0 and k_plus_2 != 0:
                        self.model += (self.w_d[i, k] + self.w_d[i, k_plus_2] <= 1, f"departure_non_adjacent_{i}_{k}_{k_plus_2}")

        # Constraint (25 in MILP): Charging time (adjusted for non-linear charging)
        for i in self.F_rep:
            departure_time = pl.lpSum([self.lambd[i, k] * self.instance.c.get((i, k), 0) for k in self.B])
            arrival_time = pl.lpSum([self.alpha[i, k] * self.instance.c.get((i, k), 0) for k in self.B])
            self.model += (self.Delta[i] == departure_time - arrival_time, f"charging_time_{i}")

        # Constraints (26-27 in MILP): Time tracking (departure time)
        for i, j in self.A:
            travel_time = self._calculate_travel_time(i, j)
            service_time = self._get_node_data(i)['service_time']
            if j in self.I:
                self.model += (self.tau[i] + (travel_time + service_time) * self.x[i, j] <= self.tau[j] + self.Tmax * (1 - self.x[i, j]), f"time_customer_{i}_{j}")
            elif j in self.F_rep:
                self.model += (self.tau[i] + self.Delta[j] + travel_time * self.x[i, j] <= self.tau[j] + (self.Tmax + self.Smax) * (1 - self.x[i, j]), f"time_station_{i}_{j}")

        # Constraints (28-29 in MILP): Time limits
        for j in self.V:
            if (j, self.E) in self.A:
                self.model += (self.tau[j] + self._calculate_travel_time(j, self.E) <= self.Tmax, f"time_to_depot_{j}")
        self.model += (self.tau[self.D] <= self.Tmax, "depot_time")

        # Constraints (30-33 in MILP): Symmetry breaking
        for i, j in self.A:
            if self.m.get((i, j), 0) == 1:
                self.model += (self.x[i, j] == 0, f"no_symmetry_arc_{i}_{j}")
        # Fixed: Iterate over pairs of charging stations correctly
        for i in self.F_rep:
            for j in self.F_rep:
                if self.m.get((i, j), 0) == 1 and j <= i:
                    self.model += (self.tau[i] >= self.tau[j], f"symmetry_time_{i}_{j}")
        for j in self.F_rep:
            self.model += (self.tau[j] <= self.Tmax * pl.lpSum([self.x[i, j] for i in self.V if (i, j) in self.A]), f"time_station_bound_{j}")
        for h in self.F_rep:
            for f in self.F_rep:
                if self.m.get((h, f), 0) == 1 and h <= f:
                    self.model += (pl.lpSum([self.x[i, h] for i in self.V if (i, h) in self.A]) >= pl.lpSum([self.x[j, f] for j in self.V if (j, f) in self.A]), f"symmetry_visit_{h}_{f}")

        # Max vehicles constraint
        vehicles_out = pl.lpSum([self.x[self.D, j] for j in self.V if (self.D, j) in self.A])
        self.model += (vehicles_out <= self.max_vehicles, "max_vehicles_constraint")

    def solve(self, time_limit: int = 3600, log_file: str = None) -> bool:
        if self.model is None:
            self.create_model()
        if os.path.exists(self.cplex_path):
            solver = pl.CPLEX_CMD(path=self.cplex_path, msg=1, timeLimit=time_limit, gapRel=0.001, logPath=log_file)
            solver_name = "CPLEX"
        else:
            solver = pl.PULP_CBC_CMD(msg=0, timeLimit=time_limit, gapRel=0.001)
            solver_name = "CBC"
        print(f"Using solver: {solver_name}")
        start_time = time.time()
        self.model.solve(solver)
        self.solve_time = time.time() - start_time
        self.solver_used = solver_name
        if self.model.status == pl.LpStatusOptimal:
            print("Solver found an optimal solution.")
            self._extract_solution()
            return True
        print(f"Solver status: {pl.LpStatus[self.model.status]}")
        self.solution = None
        return False

    def _extract_solution(self):
        try:
            num_vehicles = sum(pl.value(self.x[self.D, j]) for j in self.V if (self.D, j) in self.A)
            travel_time = sum(self._calculate_travel_time(i, j) * pl.value(self.x[i, j]) for i, j in self.A)
            service_time = sum(self._get_node_data(i)['service_time'] * sum(pl.value(self.x[j, i]) for j in self.V if (j, i) in self.A) for i in self.I)
            charging_time = sum(pl.value(self.Delta[i]) for i in self.F_rep if pl.value(self.Delta[i]) is not None and pl.value(self.Delta[i]) > 1e-4)
            total_time = travel_time + service_time + charging_time
            self.solution = {
                'num_vehicles': num_vehicles,
                'total_time': total_time,
                'travel_time': travel_time,
                'service_time': service_time,
                'charging_time': charging_time,
                'routes': self._extract_routes(),
                'battery_levels': self._extract_route_battery_levels(),
                'charging_details': self._extract_charging_details(),
                'computation_time': self.solve_time,
                'solver_used': self.solver_used,
                'mip_gap': self.mip_gap
            }
        except Exception as e:
            print(f"Warning: Could not extract solution due to: {e}")
            self.solution = None

    def _extract_routes(self) -> List[List[str]]:
        routes = []
        used_arcs = [(i, j) for i, j in self.A if pl.value(self.x[i, j]) > 0.5]
        depot_arcs = [(i, j) for i, j in used_arcs if i == self.D]
        visited = set()
        for depot_arc in depot_arcs:
            route = [self.D]
            current = depot_arc[1]
            current_arc = depot_arc
            route_arcs = [current_arc]
            while current != self.E and current not in visited:
                visited.add(current)
                route.append(current)
                next_arcs = [(i, j) for i, j in used_arcs if i == current and j not in visited]
                if not next_arcs:
                    route.append(self.E)
                    break
                current_arc = next_arcs[0]
                route_arcs.append(current_arc)
                current = current_arc[1]
            if current == self.E and route[-1] != self.E:
                route.append(self.E)
            if len(route) > 2:
                routes.append(route)
        return routes

    def _extract_route_battery_levels(self) -> Dict[str, List[Dict]]:
        battery_levels = {}
        for route in self.solution['routes']:
            route_battery = []
            for node in route:
                battery_before = pl.value(self.y_a[node]) if pl.value(self.y_a[node]) is not None else 0
                battery_after = pl.value(self.y_d[node]) if pl.value(self.y_d[node]) is not None else 0
                route_battery.append({
                    'node': node,
                    'battery_before': battery_before,
                    'battery_after': battery_after
                })
            battery_levels[f"route_{route[1]}"] = route_battery
        return battery_levels

    def _extract_charging_details(self) -> Dict[str, List[Dict]]:
        charging_details = {}
        for i in self.F_rep:
            if pl.value(self.Delta[i]) is not None and pl.value(self.Delta[i]) > 1e-4:
                charging_details[i] = [{
                    'charging_time': pl.value(self.Delta[i]),
                    'battery_before': pl.value(self.y_a[i]),
                    'battery_after': pl.value(self.y_d[i]),
                    'charge_amount': pl.value(self.y_d[i]) - pl.value(self.y_a[i])
                }]
        return charging_details

    def print_solution(self):
        if not self.solution:
            print("Không tìm thấy giải pháp khả thi!")
            return None
        print(f"\n=== Tóm tắt Giải pháp ===")
        print(f"Số lượng xe: {int(self.solution['num_vehicles']) if self.solution['num_vehicles'] is not None else '-'}")
        print(f"Tổng thời gian: {self.solution['total_time']:.2f}")
        print(f"Thời gian di chuyển: {self.solution['travel_time']:.2f}")
        print(f"Thời gian phục vụ: {self.solution['service_time']:.2f}")
        print(f"Thời gian sạc: {self.solution['charging_time']:.2f}")
        print(f"Thời gian tính toán: {self.solution['computation_time']:.2f}s")
        print(f"Trình giải: {self.solution['solver_used']}")
        if self.solution.get('mip_gap') is not None:
            print(f"MILP GAP: {self.solution['mip_gap']:.4f}")
        print(f"\n=== Tuyến đường ===")
        for i, route in enumerate(self.solution['routes']):
            print(f"Xe {i+1}: {' -> '.join(route)}")
        print(f"\n=== Mức Pin ===")
        for route_id, battery_info in self.solution['battery_levels'].items():
            print(f"{route_id}:")
            for info in battery_info:
                print(f"  Nút {info['node']}: Pin trước={info['battery_before']:.2f}, Pin sau={info['battery_after']:.2f}")
        print(f"\n=== Chi tiết Sạc ===")
        for station, details in self.solution['charging_details'].items():
            for detail in details:
                print(f"Trạm {station}: Thời gian sạc={detail['charging_time']:.2f}, Pin trước={detail['battery_before']:.2f}, Pin sau={detail['battery_after']:.2f}")
        return self.solution

In [ ]:
import pandas as pd
from pathlib import Path
import time

def solve_all_instances_enhanced():
    """Giải quyết tất cả các instance E-VRPTW với các tính năng nâng cao
    (sạc không tuyến tính, sạc một phần, tiêu thụ năng lượng phụ thuộc tải trọng)"""

    # Danh sách các instance cần giải
    instance_names = [
        "c208C5",
        "c1010C10",
        "c101C5"
    ]

    # Thư mục chứa các file instance
    data_folder = "/content/drive/MyDrive/evrptw_instances"
    results = []


    # Giải từng instance
    for instance_name in instance_names:
        instance_file = Path(data_folder) / f"{instance_name}.txt"

        print(f"\nĐang giải {instance_name} với các tính năng nâng cao...")
        print("-" * 50)

        # Kiểm tra sự tồn tại của file
        if not instance_file.exists():
            print(f"❌ Không tìm thấy file: {instance_file}")
            results.append({
                'Instance': instance_name,
                'Status': 'File Not Found',
                'Vehicles': '-',
                'Distance': '-',
                'Time': '-',
                'Total Cost': '-',
                'Solve Time': '-',
                'Solver': 'N/A',
                'Charging Details': '-'
            })
            continue

        try:
            # Phân tích instance
            instance = EVRPTWInstance()
            instance.parse_instance_file(str(instance_file))

            # Thiết lập các tham số nâng cao
            instance.w_charge_rate = 0.9  # Tốc độ sạc không dây
            instance.alpha = 2.0          # Tham số sạc không tuyến tính
            instance.beta = 0.5           # Hàm sạc căn bậc hai
            instance.set_wireless_coverage("full")  # Mức độ phủ sóng 20%

            # In thông tin instance
            print(f"📊 Thông tin instance: {len(instance.customers)} khách hàng, "
                  f"{len(instance.stations)} trạm sạc")
            print(f"⚡ Sạc không dây: {instance.w_charge_rate} đơn vị/khoảng cách với 20% phủ sóng")
            print(f"🔋 Sạc không tuyến tính: α={instance.alpha}, β={instance.beta}")

            # Đường dẫn CPLEX
            cplex_path =  "/content/drive/MyDrive/cplex/cplex"
            # solver = EnhancedEVRPTWSolver(instance, max_vehicles=3, cplex_path=cplex_path)
            solver = EnhancedEVRPTW_NL(instance, max_vehicles=3, cplex_path=cplex_path)
            # Giải instance với giới hạn thời gian
            start_time = time.time()
            success = solver.solve()
            solve_time = time.time() - start_time


            # Lấy tên solver được sử dụng
            solver_used = getattr(solver, 'solver_used', 'Unknown')

            # Thư mục lưu file LP
            lp_dir = r"/content/drive/MyDrive/EVRP-TW-DWC/lp_solver1/MILP"
            os.makedirs(lp_dir, exist_ok=True)
            lp_filename = os.path.join(lp_dir, f"{instance_name}_model.lp")
            solver.model.writeLP(lp_filename)
            print(f"📝 Đã ghi mô hình LP ra file: {lp_filename}")
            # ...existing code...
            if success and solver.solution:
                result = solver.print_solution()
                print(f"✅ ĐÃ GIẢI bằng {solver_used}")
                print(f"Thời gian: {result['time']:.6f}")
                print(f"Tổng chi phí (objective): {result['total_cost']:.6f}")
                print(f"Số xe: {result['vehicles']}")
                # ...in charging details, routes...
                charging_summary = f"{len(result['charging_details'])} phiên" if result['charging_details'] else "Không sạc"
                results.append({
                    'Instance': instance_name,
                    'Status': 'Optimal',
                    'Vehicles': result['vehicles'],
                    'Time': f"{result['time']:.6f}",
                    'Total Cost': f"{result['total_cost']:.6f}",
                    'Solve Time': f"{solve_time:.2f}s",
                    'Solver': solver_used,
                    'Charging Details': charging_summary
                })
            else:
                print(f"❌ THẤT BẠI - Không tìm thấy giải pháp (đã thử {solver_used})")
                results.append({
                    'Instance': instance_name,
                    'Status': 'Failed',
                    'Vehicles': '-',
                    'Time': '-',
                    'Total Cost': '-',
                    'Solve Time': f"{solve_time:.2f}s",
                    'Solver': solver_used,
                    'Charging Details': '-'
                })
            # ...existing code...

        except Exception as e:
            print(f"❌ LỖI: {str(e)}")
            results.append({
                'Instance': instance_name,
                'Status': 'Error',
                'Vehicles': '-',
                'Time': '-',
                'Total Cost': '-',
                'Solve Time': '-',
                'Solver': 'Error',
                'Charging Details': '-'
            })

    # In bảng tóm tắt
    print("\n" + "=" * 100)
    print("BẢNG TÓM TẮT - E-VRPTW NÂNG CAO VỚI CÁC TÍNH NĂNG TIÊN TIẾN")
    print("=" * 100)

    # Tạo DataFrame để hiển thị đẹp mắt
    df = pd.DataFrame(results)
    print(df.to_string(index=False))

    # In thống kê
    solved_instances = [r for r in results if r['Status'] == 'Optimal']
    print(f"\n📊 THỐNG KÊ:")
    print(f"Tổng số instance: {len(instance_names)}")
    print(f"Giải tối ưu thành công: {len(solved_instances)}")
    print(f"Tỷ lệ thành công: {len(solved_instances)/len(instance_names)*100:.1f}%")

    if solved_instances:
        total_vehicles = sum(int(r['Vehicles']) for r in solved_instances)
        avg_vehicles = total_vehicles / len(solved_instances)
        total_cost_sum = sum(float(r['Total Cost']) for r in solved_instances)
        avg_total_cost = total_cost_sum / len(solved_instances)
        print(f"Số xe trung bình: {avg_vehicles:.2f}")
        print(f"Tổng chi phí trung bình: {avg_total_cost:.3f}")

    # In thông tin solver và tính năng
    cplex_used = len([r for r in results if r.get('Solver') == 'CPLEX'])
    cbc_used = len([r for r in results if 'CBC' in r.get('Solver', '')])
    print(f"\n🔧 SỬ DỤNG SOLVER:")
    print(f"CPLEX: {cplex_used} instance")
    print(f"CBC (dự phòng): {cbc_used} instance")


    return results

In [ ]:
# solve_all_instances_enhanced()

In [ ]:
class OptimalChargingSolver(EnhancedEVRPTWSolver):
    """
    Một solver nhận vào một tập hợp các tuyến đường cố định và chỉ tối ưu hóa chiến lược sạc.
    Thứ tự của khách hàng và các trạm sạc là cố định. Mô hình sẽ quyết định:
    - Thời gian đến tại mỗi nút.
    - Có sạc tại một trạm trong tuyến hay không.
    - Lượng pin cần sạc (và thời gian sạc tương ứng).
    - Mức pin dọc theo tuyến đường.
    """
    def __init__(self, instance, fixed_routes, cplex_path=None):
        self.fixed_routes = fixed_routes
        # Gọi __init__ của lớp cha, nhưng sẽ override hầu hết các thiết lập của nó
        super().__init__(instance, max_vehicles=len(fixed_routes), cplex_path=cplex_path)

    def get_node_data(self, vertex_id: str) -> Dict:
        """
        Ghi đè phương thức này để xử lý các tên node duy nhất (ví dụ: 'S15_r0_s4').
        """
        if vertex_id == self.D or vertex_id == self.E:
            return self.instance.nodes[self.instance.depot_id]

        # Tách tên gốc ra khỏi hậu tố _rX_sY
        original_id = vertex_id.split('_')[0]

        if original_id in self.instance.nodes:
            return self.instance.nodes[original_id]
        else:
            # Fallback cho trường hợp tên không khớp (dù không nên xảy ra)
            return super().get_node_data(vertex_id)
    # ==================================================================

    def _setup_problem_structure(self):
        """Ghi đè phương thức của lớp cha để xây dựng mô hình dựa trên các tuyến đường cố định."""
        self.I_orig = self.instance.customers
        self.F_orig = self.instance.stations

        self.I = list(set(node for route in self.fixed_routes for node in route if node in self.I_orig))
        self.F_rep = [] # Các trạm sạc nhân bản (sẽ là duy nhất cho mỗi điểm dừng)

        self.D = f"{self.instance.depot_id}_start"
        self.E = f"{self.instance.depot_id}_end"

        self.V = {self.D, self.E}
        self.path_arcs = []

        # Tạo các nút duy nhất cho mỗi bước của mỗi tuyến
        for r_idx, route in enumerate(self.fixed_routes):
            path = [n for n in route if n != self.instance.depot_id]
            if not path: continue

            prev_unique_node = self.D
            for s_idx, node_id in enumerate(path):
                unique_node_id = f"{node_id}_r{r_idx}_s{s_idx}"
                self.V.add(unique_node_id)
                self.path_arcs.append((prev_unique_node, unique_node_id))

                if node_id in self.F_orig:
                    self.F_rep.append(unique_node_id)

                prev_unique_node = unique_node_id
            self.path_arcs.append((prev_unique_node, self.E))

        self.V = list(self.V)
        self.A = self.path_arcs

        self._setup_fixed_route_charging_breakpoints()

    def get_original_node_id(self, vertex_id: str) -> str:
        if vertex_id == self.D or vertex_id == self.E:
            return self.instance.depot_id
        return vertex_id.split('_')[0]

    def _setup_fixed_route_charging_breakpoints(self):
        """Phiên bản sửa đổi để thiết lập điểm gãy cho các trạm được ghé thăm."""
        self.instance.num_breakpoints = 2
        self.B = list(range(self.instance.num_breakpoints + 1))
        self.instance.a = {}
        self.instance.c = {}

        for station_rep_id in self.F_rep:
            for k in self.B:
                self.instance.a[station_rep_id, k] = (k / self.instance.num_breakpoints) * self.instance.Q
                if k == 0:
                    self.instance.c[station_rep_id, k] = 0.0
                else:
                    charge_amount = self.instance.a[station_rep_id, k]
                    alpha, beta = 2.0, 0.1
                    self.instance.c[station_rep_id, k] = alpha * math.sqrt(charge_amount) + beta * charge_amount

    def _create_variables(self):
        """Ghi đè để loại bỏ các biến định tuyến 'x'."""
        super()._create_variables()
        self.x = {}

    def _create_objective(self):
        """Ghi đè để tối thiểu hóa tổng thời gian, với định tuyến cố định."""
        travel_time_const = sum(self.calculate_travel_time(self.get_original_node_id(i), self.get_original_node_id(j)) for i, j in self.A)
        service_time_const = sum(self.get_node_data(node_id)['service_time'] for node_id in self.V if self.get_original_node_id(node_id) in self.I_orig)
        charging_time_var = pl.lpSum([self.Delta.get(i, 0) for i in self.F_rep])
        self.model += travel_time_const + service_time_const + charging_time_var


    def _create_constraints(self):
      # 1. Khởi tạo thời gian tại depot xuất phát
      self.model += (self.tau[self.D] == 0, "depot_start_time")
      # 2. Ràng buộc thời gian di chuyển, phục vụ, sạc giữa các node liên tiếp trên tuyến cố định
      for i, j in self.A:
          travel_time = self.calculate_travel_time(self.get_original_node_id(i), self.get_original_node_id(j))
          service_time = self.get_node_data(i)['service_time'] if self.get_original_node_id(i) in self.I_orig else 0
          charging_time = self.Delta.get(i, 0)
          # Ràng buộc: thời gian đến node j ≥ thời gian rời node i + thời gian phục vụ + thời gian sạc + thời gian di chuyển
          self.model += (self.tau[i] + service_time + charging_time + travel_time <= self.tau[j], f"time_feasibility_{i}_{j}")
      # 3. Ràng buộc cửa sổ thời gian tại mỗi node
      for j_unique in self.V:
          node_data = self.get_node_data(j_unique)
          # Thời gian đến node ≥ thời gian sớm nhất
          self.model += (self.tau[j_unique] >= node_data['ready_time'], f"time_window_early_{j_unique}")
          # Thời gian đến node ≤ thời gian muộn nhất
          self.model += (self.tau[j_unique] <= node_data['due_date'], f"time_window_late_{j_unique}")


      # 4. Khởi tạo tải xe tại depot xuất phát
      self.model += (self.u[self.D] == self.instance.C, "initial_load")


      # 5. Ràng buộc tải xe tại mỗi node trên tuyến
      for i, j in self.A:
          demand_j = self.get_node_data(j)['demand']
          if self.get_original_node_id(j) in self.I_orig:
              # Nếu node j là khách hàng: tải xe giảm đi đúng bằng nhu cầu khách hàng
              self.model += (self.u[j] == self.u[i] - demand_j, f"load_feasibility_{i}_{j}")
          elif j != self.D and j != self.E:
              # Nếu là trạm sạc hoặc node trung gian: tải xe không đổi
              self.model += (self.u[j] == self.u[i], f"load_continuity_{i}_{j}")


      # 6. Ràng buộc năng lượng (pin) khi di chuyển giữa các node
      for i, j in self.A:
          if j == self.D: continue
          distance = self.calculate_distance(self.get_original_node_id(i), self.get_original_node_id(j))
          wireless_coverage = self.get_wireless_coverage(self.get_original_node_id(i), self.get_original_node_id(j))
          consumption = self.instance.r * distance
          wireless_charge = self.instance.w * distance * wireless_coverage
          # Mức pin khi đến node j = mức pin rời node i - tiêu hao + sạc không dây trên cung (i, j)
          self.model += (self.y_a[j] == self.y_d[i] - consumption + wireless_charge, f"battery_management_{i}_{j}")


      # 7. Ràng buộc mức pin liên tục tại các node không phải trạm sạc (không sạc tại đó)
      for j in self.V:
          if j not in self.F_rep:
              # Mức pin rời node = mức pin đến node (không sạc)
              self.model += (self.y_d[j] == self.y_a[j], f"battery_continuity_{j}")


      # 8. Khởi tạo mức pin tại depot xuất phát
      self.model += (self.y_d[self.D] == self.instance.Q, "initial_battery")


      # 9. Ràng buộc sạc nonlinear tại các trạm sạc
      self._add_piecewise_charging_constraints()

    # Sạc non-linear
    def _add_piecewise_charging_constraints(self):
      for i in self.F_rep:
          # 1. Ràng buộc: chỉ được chọn đúng 1 đoạn (segment) sạc khi đến trạm (arrival)
          self.model += (pl.lpSum([self.w_a[i, k] for k in self.B if k != 0]) == self.z[i], f"arrival_segment_limit_{i}")

          # 2. Ràng buộc: chỉ được chọn đúng 1 đoạn (segment) sạc khi rời trạm (departure)
          self.model += (pl.lpSum([self.w_d[i, k] for k in self.B if k != 0]) ==  self.z[i], f"departure_segment_limit_{i}")

          # 3. Ràng buộc: mức pin khi đến trạm là tổ hợp lồi của các điểm gãy (breakpoint) (interpolation)
          self.model += (self.y_a[i] == pl.lpSum([self.alpha[i, k] * self.instance.a[i, k] for k in self.B]), f"arr_interp_{i}")

          # 4. Ràng buộc: tổng trọng số tổ hợp lồi khi đến trạm = 1 nếu trạm được ghé (chuẩn hóa)
          self.model += (pl.lpSum([self.alpha[i, k] for k in self.B]) == self.z[i], f"arr_norm_{i}")

          # 5. Ràng buộc: mức pin khi rời trạm là tổ hợp lồi của các điểm gãy (breakpoint) (interpolation)
          self.model += (self.y_d[i] == pl.lpSum([self.lambd[i, k] * self.instance.a[i, k] for k in self.B]), f"dep_interp_{i}")

          # 6. Ràng buộc: tổng trọng số tổ hợp lồi khi rời trạm = 1 nếu trạm được ghé (chuẩn hóa)
          self.model += (pl.lpSum([self.lambd[i, k] for k in self.B]) == self.z[i], f"dep_norm_{i}")

          # 7. Ràng buộc: thời gian sạc tại trạm = thời gian đạt mức pin rời trạm - thời gian đạt mức pin đến trạm
          dep_time = pl.lpSum([self.lambd[i, k] * self.instance.c[i, k] for k in self.B])
          arr_time = pl.lpSum([self.alpha[i, k] * self.instance.c[i, k] for k in self.B])
          self.model += (self.Delta[i] >= dep_time - arr_time, f"chg_time_calc_lower_{i}")

          self.model += (self.Delta[i] <= dep_time - arr_time + (1 - self.z[i]) * 1e5, f"chg_time_calc_upper_{i}")
          # 8. Ràng buộc: thời gian sạc không âm và chỉ có thể lớn hơn 0 nếu trạm được ghé
          self.model += (self.Delta[i] >= 0)
          self.model += (self.Delta[i] <= self.z[i] * 1e5)

          # 9. Ràng buộc: mức pin khi đến trạm ≤ mức pin khi rời trạm (không được sạc âm)
          self.model += (self.y_a[i] <= self.y_d[i], f"battery_non_decreasing_{i}")

    def _extract_solution(self):
        num_vehicles = len(self.fixed_routes)
        total_time = pl.value(self.model.objective)
        total_cost = 10000.0 * num_vehicles + 1.0 * total_time
        self.solution = {'num_vehicles': num_vehicles, 'total_time': total_time, 'total_cost': total_cost, 'routes': self.fixed_routes, 'charging_details': self._extract_fixed_route_charging_details(), 'route_battery_levels': self._extract_fixed_route_battery_levels(), 'computation_time': self.solve_time, 'solver_used': self.solver_used}

    def _extract_fixed_route_charging_details(self):
        details = {}
        for station_id in self.F_rep:
            if self.z.get(station_id) is not None and pl.value(self.z[station_id]) > 0.5 and self.Delta.get(station_id) is not None and pl.value(self.Delta[station_id]) > 1e-4:
                details[station_id] = {'original_station': self.get_original_node_id(station_id), 'charging_time': pl.value(self.Delta[station_id]), 'battery_before': pl.value(self.y_a[station_id]), 'battery_after': pl.value(self.y_d[station_id])}
        return details

    def _extract_fixed_route_battery_levels(self):
        levels = []
        for r_idx, route in enumerate(self.fixed_routes):
            route_levels = []
            route_levels.append((self.instance.depot_id, pl.value(self.y_a[self.D]), pl.value(self.y_d[self.D])))
            path = [n for n in route if n != self.instance.depot_id]
            for s_idx, node_id in enumerate(path):
                unique_id = f"{node_id}_r{r_idx}_s{s_idx}"
                arrival_bat = pl.value(self.y_a[unique_id])
                departure_bat = pl.value(self.y_d[unique_id])
                is_station = self.get_original_node_id(unique_id) in self.F_orig
                dep_level_to_show = departure_bat if is_station and self.z.get(unique_id) is not None and pl.value(self.z[unique_id]) > 0.5 else None
                route_levels.append((node_id, arrival_bat, dep_level_to_show))
            route_levels.append((self.instance.depot_id, pl.value(self.y_a[self.E]), None))
            levels.append(route_levels)
        return levels

In [ ]:
def generate_greedy_solution_battery_aware(instance):
    """
    Generates a greedy initial solution for the EVRPTW considering battery levels,
    time windows, and inserting charging stops when needed.

    Args:
        instance: An EVRPTWInstance object.

    Returns:
        A list of routes, where each route is a list of node IDs.
    """
    routes = []
    unvisited_customers = set(instance.customers)
    nodes = instance.nodes
    depot_id = instance.depot_id
    Q = instance.Q
    r = instance.r
    v = instance.v
    omega = instance.omega

    def calculate_distance(node1_id, node2_id):
        n1 = nodes[node1_id]
        n2 = nodes[node2_id]
        return math.sqrt((n1['x'] - n2['x'])**2 + (n1['y'] - n2['y'])**2)

    def calculate_travel_time(node1_id, node2_id):
        return calculate_distance(node1_id, node2_id) / v

    def calculate_energy_consumption(node1_id, node2_id):
        dist = calculate_distance(node1_id, node2_id)
        wireless_charge = instance.w * dist * omega.get((node1_id, node2_id), 0)
        return instance.r * dist - wireless_charge

    def get_nearest_node(from_node_id, node_list, current_time, current_battery):
        nearest_node = None
        min_distance = float('inf')
        for node_id in node_list:
            dist = calculate_distance(from_node_id, node_id)
            travel_time = calculate_travel_time(from_node_id, node_id)
            arrival_time = current_time + travel_time
            node_data = nodes[node_id]

            # Check time window
            if arrival_time > node_data['due_date']:
                continue

            # Check battery level
            energy_needed = calculate_energy_consumption(from_node_id, node_id)
            if current_battery < energy_needed:
                continue

            if dist < min_distance:
                min_distance = dist
                nearest_node = node_id

        return nearest_node

    while unvisited_customers:
        current_route = [depot_id]
        current_location = depot_id
        current_time = nodes[depot_id]['ready_time']
        current_battery = Q
        route_unvisited_customers = list(unvisited_customers) # Customers reachable in this route attempt

        while True:
            # Try to find the nearest reachable customer
            next_customer = get_nearest_node(current_location, route_unvisited_customers, current_time, current_battery)

            if next_customer:
                dist_to_customer = calculate_distance(current_location, next_customer)
                travel_time_to_customer = calculate_travel_time(current_location, next_customer)
                energy_to_customer = calculate_energy_consumption(current_location, next_customer)
                customer_data = nodes[next_customer]

                arrival_time = current_time + travel_time_to_customer
                wait_time = max(0, customer_data['ready_time'] - arrival_time)
                service_time = customer_data['service_time']
                departure_time = arrival_time + wait_time + service_time

                # Check if the customer is still feasible after considering wait time
                if departure_time <= customer_data['due_date']:
                    current_route.append(next_customer)
                    current_location = next_customer
                    current_time = departure_time
                    current_battery -= energy_to_customer
                    unvisited_customers.remove(next_customer)
                    route_unvisited_customers.remove(next_customer)
                    # Re-evaluate reachable customers from the new location
                    route_unvisited_customers = list(unvisited_customers)
                    continue # Try to add another customer

            # If no customer found or adding customer is not feasible, check if we need to charge
            dist_to_depot = calculate_distance(current_location, depot_id)
            energy_needed_to_depot = calculate_energy_consumption(current_location, depot_id)

            if current_battery < energy_needed_to_depot:
                # Need to charge, find nearest reachable charging station
                nearest_station = get_nearest_node(current_location, instance.stations, current_time, current_battery)

                if nearest_station:
                    dist_to_station = calculate_distance(current_location, nearest_station)
                    travel_time_to_station = calculate_travel_time(current_location, nearest_station)
                    arrival_time_at_station = current_time + travel_time_to_station
                    station_data = nodes[nearest_station]

                    # Check station time window
                    if arrival_time_at_station <= station_data['due_date']:
                         # Simulate charging to full (for simplicity in greedy)
                         charge_time = (Q - current_battery) / instance.g # Simplified charging time calculation
                         departure_time_from_station = max(arrival_time_at_station, station_data['ready_time']) + charge_time

                         # Check if departing station is within its due date
                         if departure_time_from_station <= station_data['due_date']:
                            current_route.append(nearest_station)
                            current_location = nearest_station
                            current_time = departure_time_from_station
                            current_battery = Q # Charged to full
                            continue # Try to add a customer again from the charged station

            # If no customer can be added and no charging is feasible or needed, break the inner loop
            break

        # Finish the route by returning to the depot if not already there
        if current_location != depot_id:
             # Re-check battery to return to depot after the last stop/charge
             dist_to_depot = calculate_distance(current_location, depot_id)
             energy_needed_to_depot = calculate_energy_consumption(current_location, depot_id)

             if current_battery < energy_needed_to_depot:
                 # Need to charge before returning to depot
                 nearest_station = get_nearest_node(current_location, instance.stations, current_time, current_battery)
                 if nearest_station:
                      dist_to_station = calculate_distance(current_location, nearest_station)
                      travel_time_to_station = calculate_travel_time(current_location, nearest_station)
                      arrival_time_at_station = current_time + travel_time_to_station
                      station_data = nodes[nearest_station]

                      if arrival_time_at_station <= station_data['due_date']:
                           charge_time = (Q - current_battery) / instance.g
                           departure_time_from_station = max(arrival_time_at_station, station_data['ready_time']) + charge_time
                           if departure_time_from_station <= station_data['due_date']:
                                current_route.append(nearest_station)
                                current_location = nearest_station
                                current_time = departure_time_from_station
                                current_battery = Q # Charged to full
                           # else: # If station visit is not feasible, route might be infeasible
                                # print(f"Warning: Cannot reach depot from {current_location} even after considering charging station {nearest_station}.")
                                # break # Break if cannot return to depot

             # Add depot as the end node
             current_route.append(depot_id)


        if len(current_route) > 2: # Ensure the route has at least one customer or station visit
            routes.append(current_route)

        # If no customers were added in this route, break the outer loop to avoid infinite loop
        if not route_unvisited_customers and unvisited_customers:
             # This means no more customers can be served with the current logic
             print("Warning: Could not serve all customers with the greedy approach.")
             break


    return routes

In [ ]:
!pip show pulp
!pip show cplex
!/content/drive/MyDrive/cplex/cplex -v

Name: PuLP
Version: 3.2.1
Summary: PuLP is an LP modeler written in python. PuLP can generate MPS or LP files and call GLPK, COIN CLP/CBC, CPLEX, and GUROBI to solve linear problems.
Home-page: 
Author: J.S. Roy
Author-email: "S.A. Mitchell" <pulp@stuartmitchell.com>, Franco Peschiera <pchtsp@gmail.com>
License: MIT
Location: /usr/local/lib/python3.11/dist-packages
Requires: 
Required-by: 

Incorrect usage. Correct command syntax is:
   /content/drive/MyDrive/cplex/cplex [-i] [-f <commandfile> | -c "<command1>" "<command2>" ...]
Exiting


In [ ]:
# import pandas as pd
# import random
# import pulp as pl
# from copy import deepcopy
# import os
# import math
# import time
# from pathlib import Path
# import sys
# import datetime

# #==============================================================================
# # STEP 1: ĐẢM BẢO CÁC LỚP CỐT LÕI ĐÃ ĐƯỢC ĐỊNH NGHĨA
# # Chạy các cell chứa EVRPTWInstance, EnhancedEVRPTWSolver, và OptimalChargingSolver trước.
# #==============================================================================

# #==============================================================================
# # STEP 2: ĐỊNH NGHĨA CÁC TOÁN TỬ CHO ALNS
# #==============================================================================

# # --- TOÁN TỬ PHÁ HỦY (DESTROY OPERATOR) ---
# def destroy_random_customers(solution, instance, k):
#     if not solution or not solution.get('routes'):
#         return {'routes': [], 'removed_customers': list(instance.customers)}
#     partial_solution = deepcopy(solution)
#     customers_in_routes = [node for route in partial_solution['routes'] for node in route if node in instance.customers]
#     if not customers_in_routes: return partial_solution
#     num_to_remove = min(k, len(customers_in_routes))
#     removed = random.sample(customers_in_routes, num_to_remove)
#     new_routes = [[node for node in route if node not in removed] for route in partial_solution['routes']]
#     partial_solution['routes'] = [r for r in new_routes if any(node in instance.customers for node in r)]
#     partial_solution['removed_customers'] = removed
#     return partial_solution

# # --- TOÁN TỬ SỬA CHỮA (REPAIR OPERATOR) ---
# def repair_greedy_insert(partial_solution, instance, solver_class):
#     temp_instance = deepcopy(instance)
#     customers_to_serve = [node for route in partial_solution.get('routes', []) for node in route if node in temp_instance.customers]
#     customers_to_serve.extend(partial_solution.get('removed_customers', []))
#     if not customers_to_serve:
#         return {'routes': [], 'num_vehicles': 0, 'total_time': 0, 'total_cost': 0}
#     temp_instance.customers = list(set(customers_to_serve))
#     solver = solver_class(temp_instance, max_vehicles=instance.max_vehicles)
#     solver.create_model()
#     # Set a shorter time limit for the repair step within ALNS
#     solver.solve(time_limit=30)
#     return solver.solution

# # --- TOÁN TỬ CẢI THIỆN CỤC BỘ (LOCAL SEARCH OPERATOR) ---
# def local_search_optimal_charging(solution, instance, name, iteration_time):
#     if not solution or not solution.get('routes'):
#         return solution
#     print("    -> Applying Optimal Charging Local Search...")
#     fixed_routes = solution['routes']
#     # Get the cplex_path from the instance or a default location
#     cplex_path = getattr(instance, 'cplex_path', "/content/drive/MyDrive/cplex/cplex")
#     # Pass the cplex_path explicitly to the OptimalChargingSolver
#     charging_optimizer = OptimalChargingSolver(instance, fixed_routes, cplex_path=cplex_path)

#     # Create directory for ALNS logs if it doesn't exist
#     alns_log_dir = r"/content/drive/MyDrive/EVRP-TW-DWC/lp_solver1/ALNS/logs"
#     os.makedirs(alns_log_dir, exist_ok=True)
#     alns_log_file = os.path.join(alns_log_dir, f"{name}_{iteration_time}_charging_log.txt")

#     print(f"    -> Writing ALNS charging log to: {alns_log_file}") # Added print statement for debugging

#     # Set a shorter time limit for the optimal charging step within ALNS
#     success = charging_optimizer.solve(time_limit=30, log_file=alns_log_file)


#     # Thư mục lưu file LP
#     lp_dir = r"/content/drive/MyDrive/EVRP-TW-DWC/lp_solver1/ALNS"
#     os.makedirs(lp_dir, exist_ok=True)
#     lp_filename = os.path.join(lp_dir, f"{name}_{iteration_time}.lp")
#     try:
#         charging_optimizer.model.writeLP(lp_filename)
#         print(f"    📝 Đã ghi mô hình LP ra file: {lp_filename}")
#     except Exception as e:
#         print(f"    Warning: Could not write LP file {lp_filename}: {e}")


#     if success and charging_optimizer.solution:
#         print("    -> Charging schedule improved.")
#         # if charging_optimizer.solver_used == "CPLEX" and charging_optimizer.mip_gap is not None:
#         #     print(f"    -> MILP GAP (Optimal Charging): {charging_optimizer.mip_gap:.4f}")
#         return charging_optimizer.solution
#     else:
#         print("    -> Could not improve charging schedule, keeping original.")
#         return solution

# #==============================================================================
# # STEP 3: LỚP ADVANCED ALNS
# #==============================================================================

# class AdvancedALNS:
#     def __init__(self, instance, solver_class, name, max_iterations=100, time_limit=300, random_seed=42):
#         self.instance = instance
#         self.solver_class = solver_class
#         self.name = name
#         self.max_iterations = max_iterations
#         self.time_limit = time_limit
#         self.random_seed = random_seed
#         self.best_solution = None
#         self.best_objective = float('inf')
#         random.seed(self.random_seed)


#     def generate_initial_solution(self):
#         print("  -> Generating initial solution for ALNS...")
#         solver = self.solver_class(self.instance, max_vehicles=self.instance.max_vehicles)
#         solver.create_model()
#         # Set a shorter time limit for initial solution generation
#         solver.solve(time_limit=60  )
#         return solver.solution
#         # return generate_greedy_solution_battery_aware(self.instance)

#     def evaluate(self, solution):
#         if solution is None: return float('inf')
#         return solution.get('total_cost', float('inf'))

#     def accept(self, old_obj, new_obj, temperature):
#         if new_obj <= old_obj : return True
#         return random.random() < math.exp(-(new_obj - old_obj) / max(temperature, 1e-6))

#     def run(self):
#         print("Run ALNS")
#         start_time = time.time()
#         S = self.generate_initial_solution()
#         if not S:
#             print("  -> ALNS failed to generate an initial solution.")
#             return None
#         S_star = deepcopy(S)
#         O_current = self.evaluate(S)
#         O_star = O_current
#         print(f"  -> Initial Objective: {O_star:.2f}")
#         temperature, gamma = 100, 0.98
#         for i in range(self.max_iterations):
#             run_time = time.time() - start_time
#             if (run_time) > self.time_limit:
#                 print("  -> ALNS time limit reached.")
#                 break
#             print(f"\n--- ALNS Iteration {i} (Temp: {temperature:.2f}) ---")
#             k = random.randint(2, max(2, int(0.25 * len(self.instance.customers))))
#             print(f"  -> Destroying {k} customers...")
#             S_partial = destroy_random_customers(S, self.instance, k)
#             print("  -> Repairing solution with MILP-based insert...")
#             S_prime = repair_greedy_insert(S_partial, self.instance, self.solver_class)
#             if S_prime is None:
#                 print("  -> Repair failed. Skipping iteration.")
#                 continue

#             # Gọi ra hàm MILP sạc
#             S_prime_refined = local_search_optimal_charging(S_prime, self.instance, self.name, i)

#             O_prime = self.evaluate(S_prime_refined)
#             if self.accept(O_current, O_prime, temperature):
#                 S, O_current = deepcopy(S_prime_refined), O_prime
#                 print(f"  -> Accepted new solution. Current Objective: {O_current:.2f}")
#                 if O_prime < O_star:
#                     S_star, O_star = deepcopy(S_prime_refined), O_prime
#                     print(f"  ⭐⭐⭐ New best solution found! Best Objective: {O_star:.2f} ⭐⭐⭐")
#             else:
#                 print(f"  -> Rejected new solution with objective {O_prime:.2f}")
#             temperature *= gamma
#         self.best_solution = S_star
#         return S_star

In [ ]:
import pandas as pd
import random
import pulp as pl
from copy import deepcopy
import os
import math
import time
from pathlib import Path
import sys
import datetime
import uuid

# Hàm tính khoảng cách Euclidean giữa hai đỉnh
def calculate_distance(node1_id, node2_id, instance):
    # Giả định rằng instance có thuộc tính nodes với tọa độ x, y
    if node1_id not in instance.nodes or node2_id not in instance.nodes:
        # Handle cases where node_id might be depot_start or depot_end
        n1_id = instance.depot_id if node1_id in [f"{instance.depot_id}_start", f"{instance.depot_id}_end"] else node1_id
        n2_id = instance.depot_id if node2_id in [f"{instance.depot_id}_start", f"{instance.depot_id}_end"] else node2_id
        if n1_id not in instance.nodes or n2_id not in instance.nodes:
             print(f"Warning: Node {node1_id} or {node2_id} not found in instance nodes.")
             return float('inf') # Return a large value for infeasible distance

        x1 = instance.nodes[n1_id]['x']
        y1 = instance.nodes[n1_id]['y']
        x2 = instance.nodes[n2_id]['x']
        y2 = instance.nodes[n2_id]['y']
    else:
        x1 = instance.nodes[node1_id]['x']
        y1 = instance.nodes[node1_id]['y']
        x2 = instance.nodes[node2_id]['x']
        y2 = instance.nodes[node2_id]['y']

    return math.sqrt((x2 - x1) ** 2 + (y2 - y1) ** 2)


# Hàm tạo ma trận khoảng cách nếu không có sẵn
def compute_distance_matrix(instance):
    # Use original node IDs from instance
    nodes_list = list(instance.nodes.keys())
    n = len(nodes_list)
    distance_matrix = {}
    for i in nodes_list:
        distance_matrix[i] = {}
        for j in nodes_list:
            distance_matrix[i][j] = calculate_distance(i, j, instance)
    return distance_matrix

# --- TOÁN TỬ PHÁ HỦY ---
def destroy_random_customers(solution, instance, k):
    if not solution or not solution.get('routes'):
        return {'routes': [], 'removed_customers': list(instance.customers)}
    partial_solution = deepcopy(solution)
    # Ensure only original customer IDs are considered
    customers_in_routes = list(set([node for route in partial_solution['routes'] for node in route if node in instance.customers]))

    if not customers_in_routes:
        return partial_solution
    num_to_remove = min(k, len(customers_in_routes))
    removed = random.sample(customers_in_routes, num_to_remove)
    # Remove customers from routes
    new_routes = []
    for route in partial_solution['routes']:
        new_route = [node for node in route if node not in removed]
        # Keep depot nodes even if route is empty of customers/stations
        if len(new_route) > 1: # Keep routes with at least D -> E or D -> Node -> E
             new_routes.append(new_route)
        elif instance.depot_id in new_route and len(new_route) == 1:
             pass # Remove routes that are just a single depot node
        elif instance.depot_id in new_route and len(new_route) == 0:
             pass # Remove empty routes

    partial_solution['routes'] = new_routes
    partial_solution['removed_customers'] = removed
    return partial_solution


def destroy_worst_removal(solution, instance, k):
    if not solution or not solution.get('routes'):
        return {'routes': [], 'removed_customers': list(instance.customers)}
    partial_solution = deepcopy(solution)
    customers_in_routes = list(set([node for route in partial_solution['routes'] for node in route if node in instance.customers]))

    if not customers_in_routes:
        return partial_solution

    # Ensure distance_matrix exists in instance
    if not hasattr(instance, 'distance_matrix'):
        instance.distance_matrix = compute_distance_matrix(instance)

    removal_costs = []
    # Evaluate cost impact of removing each customer
    for customer in customers_in_routes:
        temp_solution = deepcopy(partial_solution)
        found = False
        for route in temp_solution['routes']:
            if customer in route:
                route.remove(customer)
                found = True
                break
        if found:
            temp_solution['routes'] = [r for r in temp_solution['routes'] if len(r) > 1 or (len(r) == 1 and r[0] == instance.depot_id)] # Keep valid routes
            temp_cost = evaluate_solution(temp_solution, instance)
            # The cost impact is the original cost minus the cost after removal
            original_cost = evaluate_solution(partial_solution, instance)
            cost_impact = original_cost - temp_cost
            removal_costs.append((customer, cost_impact))


    # Sort by cost impact (most positive impact first)
    removal_costs.sort(key=lambda x: x[1], reverse=True)
    removed = [x[0] for x in removal_costs[:min(k, len(removal_costs))]]

    # Remove customers from routes
    new_routes = []
    for route in partial_solution['routes']:
        new_route = [node for node in route if node not in removed]
        if len(new_route) > 1:
            new_routes.append(new_route)
        elif instance.depot_id in new_route and len(new_route) == 1:
             pass
        elif instance.depot_id in new_route and len(new_route) == 0:
             pass

    partial_solution['routes'] = new_routes
    partial_solution['removed_customers'] = removed
    return partial_solution


def destroy_route_removal(solution, instance, k):
    if not solution or not solution.get('routes'):
        return {'routes': [], 'removed_customers': list(instance.customers)}
    partial_solution = deepcopy(solution)
    if len(partial_solution['routes']) == 0:
        return partial_solution

    # Determine number of routes to remove (min of k and number of routes)
    num_routes_to_remove = min(k, len(partial_solution['routes']))

    # Select routes to remove
    routes_to_remove = random.sample(partial_solution['routes'], num_routes_to_remove)

    removed_customers = []
    new_routes = []
    for route in partial_solution['routes']:
        if route in routes_to_remove:
            removed_customers.extend([node for node in route if node in instance.customers])
        else:
            new_routes.append(route)

    partial_solution['routes'] = new_routes
    partial_solution['removed_customers'] = list(set(removed_customers)) # Ensure unique customers
    return partial_solution


def destroy_cluster_removal(solution, instance, k):
    if not solution or not solution.get('routes'):
        return {'routes': [], 'removed_customers': list(instance.customers)}
    partial_solution = deepcopy(solution)
    customers_in_routes = list(set([node for route in partial_solution['routes'] for node in route if node in instance.customers]))
    if not customers_in_routes:
        return partial_solution

    # Ensure distance_matrix exists in instance
    if not hasattr(instance, 'distance_matrix'):
        instance.distance_matrix = compute_distance_matrix(instance)

    seed_customer = random.choice(customers_in_routes)

    distances = []
    for customer in customers_in_routes:
        if customer != seed_customer:
            dist = instance.distance_matrix[seed_customer][customer]
            distances.append((customer, dist))

    distances.sort(key=lambda x: x[1])
    removed = [seed_customer] + [x[0] for x in distances[:min(k-1, len(distances))]]

    # Remove customers from routes
    new_routes = []
    for route in partial_solution['routes']:
        new_route = [node for node in route if node not in removed]
        if len(new_route) > 1:
            new_routes.append(new_route)
        elif instance.depot_id in new_route and len(new_route) == 1:
             pass
        elif instance.depot_id in new_route and len(new_route) == 0:
             pass

    partial_solution['routes'] = new_routes
    partial_solution['removed_customers'] = removed
    return partial_solution

def destroy_station_vicinity_removal(solution, instance, k):
    if not solution or not solution.get('routes'):
        return {'routes': [], 'removed_customers': list(instance.customers)}
    partial_solution = deepcopy(solution)
    customers_in_routes = list(set([node for route in partial_solution['routes'] for node in route if node in instance.customers]))
    if not customers_in_routes:
        return partial_solution

    stations = instance.stations
    if not stations:
        return partial_solution

    # Ensure distance_matrix exists in instance
    if not hasattr(instance, 'distance_matrix'):
        instance.distance_matrix = compute_distance_matrix(instance)

    seed_station = random.choice(stations)

    # Calculate a reasonable radius based on instance scale
    all_nodes_in_matrix = list(instance.distance_matrix.keys())
    max_distance = 0
    if len(all_nodes_in_matrix) > 1:
        max_distance = max(instance.distance_matrix[i][j] for i in all_nodes_in_matrix for j in all_nodes_in_matrix)

    radius = random.uniform(0.1, 0.5) * max_distance if max_distance > 0 else 100 # Default radius if max_distance is 0

    removed = []
    for customer in customers_in_routes:
        if instance.distance_matrix[seed_station][customer] <= radius:
            removed.append(customer)
        if len(removed) >= k:
            break

    # Remove customers from routes
    new_routes = []
    for route in partial_solution['routes']:
        new_route = [node for node in route if node not in removed]
        if len(new_route) > 1:
            new_routes.append(new_route)
        elif instance.depot_id in new_route and len(new_route) == 1:
             pass
        elif instance.depot_id in new_route and len(new_route) == 0:
             pass

    partial_solution['routes'] = new_routes
    partial_solution['removed_customers'] = removed
    return partial_solution

def destroy_related_removal(solution, instance, k):
    if not solution or not solution.get('routes'):
        return {'routes': [], 'removed_customers': list(instance.customers)}
    partial_solution = deepcopy(solution)
    customers_in_routes = list(set([node for route in partial_solution['routes'] for node in route if node in instance.customers]))
    if not customers_in_routes:
        return partial_solution

    # Ensure distance_matrix and time_windows exist in instance
    if not hasattr(instance, 'distance_matrix'):
        instance.distance_matrix = compute_distance_matrix(instance)
    if not hasattr(instance, 'time_windows'):
         # Assuming time_windows can be derived from node data if not directly available
         instance.time_windows = {node_id: (data['ready_time'], data['due_date']) for node_id, data in instance.nodes.items()}


    relatedness_matrix = build_relatedness_matrix(instance)
    nodes_for_relatedness = instance.customers + instance.stations
    node_to_index = {node: idx for idx, node in enumerate(nodes_for_relatedness)}

    seed_customer = random.choice(customers_in_routes)
    if seed_customer not in node_to_index:
        # Handle case where seed customer is not in the nodes used for relatedness matrix
        return destroy_random_customers(solution, instance, k) # Fallback to random removal


    seed_idx = node_to_index[seed_customer]

    related = []
    for customer in customers_in_routes:
        if customer != seed_customer and customer in node_to_index:
            related_idx = node_to_index[customer]
            related.append((customer, relatedness_matrix[seed_idx][related_idx]))

    related.sort(key=lambda x: x[1], reverse=True)
    removed = [seed_customer] + [x[0] for x in related[:min(k-1, len(related))]]

    # Remove customers from routes
    new_routes = []
    for route in partial_solution['routes']:
        new_route = [node for node in route if node not in removed]
        if len(new_route) > 1:
            new_routes.append(new_route)
        elif instance.depot_id in new_route and len(new_route) == 1:
             pass
        elif instance.depot_id in new_route and len(new_route) == 0:
             pass

    partial_solution['routes'] = new_routes
    partial_solution['removed_customers'] = removed
    return partial_solution

def destroy_time_window_removal(solution, instance, k):
    if not solution or not solution.get('routes'):
        return {'routes': [], 'removed_customers': list(instance.customers)}
    partial_solution = deepcopy(solution)
    customers_in_routes = list(set([node for route in partial_solution['routes'] for node in route if node in instance.customers]))
    if not customers_in_routes:
        return partial_solution

    # Ensure time_windows exists in instance
    if not hasattr(instance, 'time_windows'):
         instance.time_windows = {node_id: (data['ready_time'], data['due_date']) for node_id, data in instance.nodes.items()}


    seed_customer = random.choice(customers_in_routes)
    if seed_customer not in instance.time_windows:
        # Handle case where seed customer is not in time_windows
         return destroy_random_customers(solution, instance, k)

    seed_time_window_start = instance.time_windows[seed_customer][0]

    time_diffs = []
    for customer in customers_in_routes:
        if customer != seed_customer and customer in instance.time_windows:
            time_diff = abs(instance.time_windows[customer][0] - seed_time_window_start)
            time_diffs.append((customer, time_diff))

    time_diffs.sort(key=lambda x: x[1])
    removed = [seed_customer] + [x[0] for x in time_diffs[:min(k-1, len(time_diffs))]]

    # Remove customers from routes
    new_routes = []
    for route in partial_solution['routes']:
        new_route = [node for node in route if node not in removed]
        if len(new_route) > 1:
            new_routes.append(new_route)
        elif instance.depot_id in new_route and len(new_route) == 1:
             pass
        elif instance.depot_id in new_route and len(new_route) == 0:
             pass

    partial_solution['routes'] = new_routes
    partial_solution['removed_customers'] = removed
    return partial_solution


def build_relatedness_matrix(instance):
    # Use original customer and station IDs
    nodes_for_relatedness = instance.customers + instance.stations
    n = len(nodes_for_relatedness)
    # Ensure distance_matrix and time_windows exist in instance
    if not hasattr(instance, 'distance_matrix'):
        instance.distance_matrix = compute_distance_matrix(instance)
    if not hasattr(instance, 'time_windows'):
         instance.time_windows = {node_id: (data['ready_time'], data['due_date']) for node_id, data in instance.nodes.items()}

    matrix = [[0.0] * n for _ in range(n)]
    for i, node_i in enumerate(nodes_for_relatedness):
        for j, node_j in enumerate(nodes_for_relatedness):
            if i != j:
                # Ensure nodes exist in the distance matrix and time windows before accessing
                if node_i in instance.distance_matrix and node_j in instance.distance_matrix[node_i] and \
                   node_i in instance.time_windows and node_j in instance.time_windows:
                    dist = instance.distance_matrix[node_i][node_j]
                    time_diff = abs(instance.time_windows[node_i][0] - instance.time_windows[node_j][0])
                    matrix[i][j] = 1.0 / (dist + time_diff + 1e-6)
                else:
                     # Assign a low relatedness if data is missing for a pair
                     matrix[i][j] = 1e-6
            else:
                matrix[i][j] = 0.0 # A node is not related to itself
    return matrix

# --- TOÁN TỬ SỬA CHỮA ---
def repair_greedy_insert(partial_solution, instance, solver_class):
    # This repair operator uses the MILP solver to build a solution from the remaining customers
    temp_instance = deepcopy(instance)
    customers_to_serve = [node for route in partial_solution.get('routes', []) for node in route if node in temp_instance.customers]
    customers_to_serve.extend(partial_solution.get('removed_customers', []))
    customers_to_serve = list(set(customers_to_serve)) # Ensure unique customers

    if not customers_to_serve:
         # If no customers to serve, return an empty solution with depot
         return {'routes': [[instance.depot_id, instance.depot_id]], 'num_vehicles': 0, 'total_time': 0, 'total_cost': 0}

    # Create a temporary instance with only the customers to be served
    temp_instance.customers = customers_to_serve
    # Filter out stations not present in the original instance (if any were added incorrectly)
    temp_instance.stations = [s for s in temp_instance.stations if s in instance.stations]
    # Ensure depot is in temp_instance nodes
    if instance.depot_id not in temp_instance.nodes:
        temp_instance.nodes[instance.depot_id] = instance.nodes[instance.depot_id]


    # Use the specified solver class (either EnhancedEVRPTWSolver or EnhancedEVRPTW_NL)
    solver = solver_class(temp_instance, max_vehicles=instance.max_vehicles) # Use max_vehicles from original instance
    solver.create_model()

    # Set a shorter time limit for the repair step within ALNS
    # Also, create a log file for the repair step
    repair_log_dir = r"/content/drive/MyDrive/EVRP-TW-DWC/lp_solver1/ALNS/repair_logs"
    os.makedirs(repair_log_dir, exist_ok=True)
    repair_log_file = os.path.join(repair_log_dir, f"repair_{uuid.uuid4()}.txt") # Use a unique ID for each log file

    print(f"    -> Running MILP repair with log: {repair_log_file}")
    success = solver.solve(time_limit=30, log_file=repair_log_file)

    if success and solver.solution:
        print("    -> MILP repair successful.")
        # The MILP solver returns a full solution including routes, battery, etc.
        # We need to format it to match the ALNS solution structure
        repaired_solution = {
            'routes': solver.solution.get('routes', []),
            'num_vehicles': solver.solution.get('num_vehicles', 0),
            'total_time': solver.solution.get('total_time', 0),
            'total_cost': solver.solution.get('total_cost', 0),
            'battery_levels': solver.solution.get('route_battery_levels', []), # Assuming this key exists
            'charging_details': solver.solution.get('charging_details', {}), # Assuming this key exists
            'computation_time': solver.solution.get('computation_time', 0),
            'solver_used': solver.solution.get('solver_used', 'MILP'),
            'mip_gap': solver.solution.get('mip_gap', None) # Assuming this key exists
        }
        return repaired_solution
    else:
        print("    -> MILP repair failed.")
        # If repair fails, return a solution with the remaining customers as removed
        # and potentially empty routes, or the original partial solution if preferred.
        # Here, let's return the partial solution with removed customers, indicating failure to re-insert.
        return partial_solution


def repair_best_insert(partial_solution, instance, solver_class):
    # This repair operator inserts removed customers back into existing routes
    # or new routes based on the best cost insertion. This does NOT use the MILP solver
    # to find the optimal routes, but rather evaluates insertions based on a cost function
    # (e.g., distance or a simplified objective).

    customers_to_serve = partial_solution.get('removed_customers', [])
    if not customers_to_serve:
        return partial_solution

    # Ensure distance_matrix exists in instance
    if not hasattr(instance, 'distance_matrix'):
        instance.distance_matrix = compute_distance_matrix(instance)

    # Create a mutable copy of the routes
    current_routes = deepcopy(partial_solution.get('routes', []))
    if not current_routes:
         # If no routes exist, start new routes for each customer (simple approach)
         current_routes = [[instance.depot_id, customer, instance.depot_id] for customer in customers_to_serve]
         partial_solution['routes'] = current_routes
         partial_solution['removed_customers'] = []
         # Re-evaluate the solution after insertion
         partial_solution['total_cost'] = evaluate_solution(partial_solution, instance)
         partial_solution['num_vehicles'] = len(partial_solution['routes'])
         return partial_solution


    # Try inserting each removed customer
    for customer in customers_to_serve:
        best_insertion = None # (route_idx, position, cost_increase)
        best_cost_increase = float('inf')

        # Option 1: Insert into an existing route
        for r_idx, route in enumerate(current_routes):
            # Try inserting at each possible position (between nodes)
            for pos in range(1, len(route)): # Exclude inserting before the start depot
                # Create a temporary route by inserting the customer
                temp_route = route[:pos] + [customer] + route[pos:]

                # Calculate the cost increase of this insertion
                # Cost of new edges - Cost of old edge
                cost_increase = (instance.distance_matrix[route[pos-1]][customer] +
                                 instance.distance_matrix[customer][route[pos]] -
                                 instance.distance_matrix[route[pos-1]][route[pos]])

                # Consider time window and capacity feasibility (simplified check)
                # A more sophisticated check would be needed for a full implementation
                is_feasible = True # Assume feasible for simplicity

                if is_feasible and cost_increase < best_cost_increase:
                    best_cost_increase = cost_increase
                    best_insertion = (r_idx, pos, customer)

        # Option 2: Insert into a new route (Depot -> Customer -> Depot)
        cost_new_route = (instance.distance_matrix[instance.depot_id][customer] +
                          instance.distance_matrix[customer][instance.depot_id])
        if cost_new_route < best_cost_increase:
             best_cost_increase = cost_new_route
             best_insertion = (len(current_routes), 1, customer) # Insert as a new route

        # Perform the best insertion
        if best_insertion:
            r_idx, pos, inserted_customer = best_insertion
            if r_idx == len(current_routes): # New route
                current_routes.append([instance.depot_id, inserted_customer, instance.depot_id])
            else: # Existing route
                current_routes[r_idx].insert(pos, inserted_customer)

    partial_solution['routes'] = current_routes
    partial_solution['removed_customers'] = [] # All removed customers are attempted to be inserted
    # Re-evaluate the solution after insertion
    partial_solution['total_cost'] = evaluate_solution(partial_solution, instance)
    partial_solution['num_vehicles'] = len(partial_solution['routes']) # Update vehicle count
    return partial_solution


def repair_random_insert(partial_solution, instance, solver_class):
    # This repair operator inserts removed customers randomly into existing routes
    # or new routes.

    customers_to_serve = partial_solution.get('removed_customers', [])
    if not customers_to_serve:
        return partial_solution

    current_routes = deepcopy(partial_solution.get('routes', []))

    for customer in customers_to_serve:
        if not current_routes or random.random() < 0.3: # 30% chance to start a new route
            current_routes.append([instance.depot_id, customer, instance.depot_id])
        else:
            route_idx = random.randint(0, len(current_routes) - 1)
            route = current_routes[route_idx]
            # Insert at a random position between the start depot and the end depot
            insert_pos = random.randint(1, max(1, len(route) - 1))
            route.insert(insert_pos, customer)

    partial_solution['routes'] = current_routes
    partial_solution['removed_customers'] = [] # All removed customers are attempted to be inserted
    # Re-evaluate the solution after insertion
    partial_solution['total_cost'] = evaluate_solution(partial_solution, instance)
    partial_solution['num_vehicles'] = len(partial_solution['routes']) # Update vehicle count
    return partial_solution


# --- HÀM ĐÁNH GIÁ ---
def evaluate_solution(solution, instance):
    """
    Evaluates a solution based on the total cost objective function.
    This should align with the objective function of the MILP solver.
    Objective: M1 * num_vehicles + M2 * total_time
    where total_time = travel_time + service_time + charging_time.
    Since ALNS works on routes, we need to estimate these components from the route structure.

    Args:
        solution (dict): A dictionary representing the solution, expected to have 'routes'.
        instance (EVRPTWInstance): The problem instance.

    Returns:
        float: The evaluated cost of the solution.
    """
    if not solution or not solution.get('routes'):
        # Consider an empty solution to have a very high cost (or based on max vehicles)
        return 10000.0 * instance.max_vehicles # High penalty for no solution/empty routes


    # Use constants from the MILP objective function
    M1 = 10000.0  # Penalty for each vehicle
    M2 = 1.0      # Weight for total time

    num_vehicles = len(solution['routes'])

    # Estimate travel time from routes
    travel_time = 0
    # Ensure distance_matrix exists in instance
    if not hasattr(instance, 'distance_matrix'):
        instance.distance_matrix = compute_distance_matrix(instance)
    # Ensure instance.v (average velocity) exists
    v = getattr(instance, 'v', 1.0) # Default velocity if not specified

    for route in solution['routes']:
        for i in range(len(route) - 1):
            node1_id = route[i]
            node2_id = route[i+1]
            # Use original node IDs for distance calculation
            orig_node1_id = node1_id.split('_r')[0] if '_r' in node1_id else node1_id
            orig_node2_id = node2_id.split('_r')[0] if '_r' in node2_id else node2_id

            # Handle depot start/end nodes by mapping back to original depot ID
            if orig_node1_id == f"{instance.depot_id}_start" or orig_node1_id == f"{instance.depot_id}_end":
                 orig_node1_id = instance.depot_id
            if orig_node2_id == f"{instance.depot_id}_start" or orig_node2_id == f"{instance.depot_id}_end":
                 orig_node2_id = instance.depot_id

            try:
                 dist = instance.distance_matrix[orig_node1_id][orig_node2_id]
                 travel_time += dist / v
            except KeyError:
                 print(f"Warning: Distance not found between {orig_node1_id} and {orig_node2_id} in distance matrix.")
                 # Penalize missing distances heavily to discourage infeasible routes
                 travel_time += 1e6


    # Estimate service time from routes
    service_time = 0
    # Ensure instance.nodes has service_time
    for route in solution['routes']:
        for node_id in route:
            # Use original node ID for service time
            orig_node_id = node_id.split('_r')[0] if '_r' in node_id else node_id
            if orig_node_id in instance.nodes and instance.nodes[orig_node_id]['type'] == 'c':
                 service_time += instance.nodes[orig_node_id]['service_time']


    # Estimate charging time
    # This is the most difficult to estimate accurately without a full charging model.
    # A simplification is to assume a fixed charging time per station visit,
    # or rely on the output of the OptimalChargingSolver if it was run.
    # For a basic ALNS evaluation, we can ignore it or use a simple heuristic.
    # Since we have the OptimalChargingSolver step, we can potentially use its output
    # if the solution object contains 'charging_details'.
    charging_time = 0
    if 'charging_details' in solution and solution['charging_details']:
        # Sum up the reported charging times from the optimal charging step
        for station_details in solution['charging_details'].values():
             # charging_details can be a list of visits per station in the original solver
             # In the fixed route solver, it's a dict per unique station visit
             if isinstance(station_details, list):
                  for detail in station_details:
                       charging_time += detail.get('charging_time', 0)
             elif isinstance(station_details, dict): # From fixed route solver
                  charging_time += station_details.get('charging_time', 0)
    # else:
        # Fallback: Simple heuristic for charging time if no charging details available
        # Count station visits and multiply by average charging time (if known)
        # For now, let's assume charging time is negligible in the initial evaluation if not calculated
        # or use a small penalty per station visit if needed.
        # Let's skip heuristic for now and rely on OptimalChargingSolver for accuracy.
        pass


    total_time = travel_time + service_time + charging_time

    total_cost = M1 * num_vehicles + M2 * total_time

    return total_cost


# --- TOÁN TỬ CẢI THIỆN CỤC BỘ ---
def local_search_optimal_charging(solution, instance, name, iteration_time):
    if not solution or not solution.get('routes'):
        print("    -> No solution or routes for Optimal Charging Local Search.")
        return solution
    print("    -> Applying Optimal Charging Local Search...")
    fixed_routes = solution['routes']

    # Check if fixed_routes are valid (e.g., not empty or just depots)
    if not fixed_routes or all(len(route) <= 1 for route in fixed_routes):
         print("    -> Fixed routes are invalid or empty, skipping Optimal Charging.")
         return solution


    cplex_path = getattr(instance, 'cplex_path', "/content/drive/MyDrive/cplex/cplex")
    # Pass the cplex_path explicitly to the OptimalChargingSolver
    charging_optimizer = OptimalChargingSolver(instance, fixed_routes, cplex_path=cplex_path)

    # Create directory for ALNS logs if it doesn't exist
    alns_log_dir = r"/content/drive/MyDrive/EVRP-TW-DWC/lp_solver1/ALNS/logs"
    os.makedirs(alns_log_dir, exist_ok=True)
    alns_log_file = os.path.join(alns_log_dir, f"{name}_{iteration_time}_charging_log.txt")

    print(f"    -> Writing ALNS charging log to: {alns_log_file}") # Added print statement for debugging

    # Set a shorter time limit for the optimal charging step within ALNS
    success = charging_optimizer.solve(time_limit=30, log_file=alns_log_file)


    # Thư mục lưu file LP
    lp_dir = r"/content/drive/MyDrive/EVRP-TW-DWC/lp_solver1/ALNS"
    os.makedirs(lp_dir, exist_ok=True)
    # Use a unique identifier for the LP file name within ALNS iterations
    lp_filename = os.path.join(lp_dir, f"{name}_iter{iteration_time}_charging.lp")
    try:
        if charging_optimizer.model:
            charging_optimizer.model.writeLP(lp_filename)
            print(f"    📝 Đã ghi mô hình LP ra file: {lp_filename}")
        else:
            print(f"    Warning: Optimal Charging model is None for {lp_filename}")
    except Exception as e:
        print(f"    Warning: Could not write LP file {lp_filename}: {e}")


    if success and charging_optimizer.solution:
        print("    -> Charging schedule improved.")
        # The OptimalChargingSolver returns a solution dictionary.
        # We need to merge its details back into the main ALNS solution structure.
        # The routes themselves are fixed, but timing, battery, and charging details are updated.
        improved_solution = deepcopy(solution) # Start with the current ALNS solution structure
        improved_solution['total_time'] = charging_optimizer.solution.get('total_time', improved_solution.get('total_time', 0))
        improved_solution['total_cost'] = charging_optimizer.solution.get('total_cost', evaluate_solution(improved_solution, instance)) # Recalculate if not provided
        improved_solution['battery_levels'] = charging_optimizer.solution.get('route_battery_levels', improved_solution.get('battery_levels', []))
        improved_solution['charging_details'] = charging_optimizer.solution.get('charging_details', improved_solution.get('charging_details', {}))
        improved_solution['computation_time'] = charging_optimizer.solution.get('computation_time', improved_solution.get('computation_time', 0)) # Add charging time
        improved_solution['solver_used'] = charging_optimizer.solution.get('solver_used', improved_solution.get('solver_used', 'OptimalCharging'))
        improved_solution['mip_gap'] = charging_optimizer.solution.get('mip_gap', improved_solution.get('mip_gap', None))


        return improved_solution
    else:
        print("    -> Could not improve charging schedule, keeping original.")
        return solution

# --- LỚP ADVANCED ALNS ---
class AdvancedALNS:
    def __init__(self, instance, solver_class, name, max_iterations=100, time_limit=3600, random_seed=42):
        self.instance = instance
        self.solver_class = solver_class # The MILP solver class to use for repair
        self.name = name
        self.max_iterations = max_iterations
        self.time_limit = time_limit
        self.random_seed = random_seed
        self.best_solution = None
        self.best_objective = float('inf')
        random.seed(self.random_seed)

        # Pre-compute distance matrix and time windows for faster access
        if not hasattr(self.instance, 'distance_matrix'):
             self.instance.distance_matrix = compute_distance_matrix(self.instance)
        if not hasattr(self.instance, 'time_windows'):
             self.instance.time_windows = {node_id: (data['ready_time'], data['due_date']) for node_id, data in self.instance.nodes.items()}


        # Khởi tạo danh sách toán tử với trọng số
        self.destroy_operators = [
            (destroy_random_customers, 1.0, "Random Removal"),
            (destroy_worst_removal, 1.0, "Worst Removal"),
            (destroy_route_removal, 1.0, "Route Removal"),
            (destroy_cluster_removal, 1.0, "Cluster Removal"),
            (destroy_station_vicinity_removal, 1.0, "Station Vicinity Removal"),
            (destroy_related_removal, 1.0, "Related Removal"),
            (destroy_time_window_removal, 1.0, "Time Window Removal")
        ]
        self.repair_operators = [
            # Using MILP-based repair for initial solution generation,
            # but potentially using simpler repair operators within ALNS iterations
            (repair_greedy_insert, 1.0, "MILP Greedy Insert"), # Using MILP for initial insert
            (repair_best_insert, 1.0, "Best Insert"),
            (repair_random_insert, 1.0, "Random Insert")
        ]

    def generate_initial_solution(self):
        print("  -> Generating initial solution for ALNS...")
        # Use the greedy battery-aware approach first
        greedy_routes = generate_greedy_solution_battery_aware(self.instance)

        if not greedy_routes:
             print("  -> Greedy initial solution generation failed.")
             # Fallback to MILP for a small set of customers if greedy fails completely
             if len(self.instance.customers) < 20: # Arbitrary threshold
                 print("  -> Falling back to MILP for initial solution (small instance)...")
                 solver = self.solver_class(self.instance, max_vehicles=self.instance.max_vehicles)
                 solver.create_model()
                 solver.solve(time_limit=120) # Give MILP a bit more time for initial solution
                 return solver.solution
             else:
                  return None # Cannot generate initial solution

        # If greedy routes are generated, evaluate them and create a solution dict
        initial_solution = {
            'routes': greedy_routes,
            'removed_customers': [], # No removed customers initially
            'num_vehicles': len(greedy_routes),
            # The following will be estimated by evaluate_solution or calculated by OptimalChargingSolver
            'total_time': 0,
            'total_cost': 0,
            'battery_levels': [],
            'charging_details': {},
            'computation_time': 0,
            'solver_used': 'Greedy',
            'mip_gap': None
        }

        # Evaluate the greedy solution
        initial_solution['total_cost'] = evaluate_solution(initial_solution, self.instance)

        # Optionally, run the optimal charging solver on the initial greedy routes
        print("  -> Optimizing charging for the initial greedy solution...")
        initial_solution = local_search_optimal_charging(initial_solution, self.instance, f"{self.name}_initial", "greedy")


        return initial_solution


    def evaluate(self, solution):
        """
        Evaluates a solution using the defined objective function calculation.
        """
        return evaluate_solution(solution, self.instance)

    def accept(self, old_obj, new_obj, temperature):
        if new_obj <= old_obj : return True
        return random.random() < math.exp(-(new_obj - old_obj) / max(temperature, 1e-6))

    def update_weights(self, operator_list, used_op_idx, score, segment_scores):
        """Update operator weights based on performance in the last segment."""
        rho = 0.1 # Reaction factor
        for i, (op, weight, name) in enumerate(operator_list):
            new_weight = (1 - rho) * weight + rho * segment_scores.get(name, 0) / max(1, sum(segment_scores.values())) # Normalize segment score
            operator_list[i] = (op, new_weight, name)


    def select_operator(self, operator_list):
        total_weight = sum(weight for _, weight, _ in operator_list)
        if total_weight == 0: # Prevent division by zero if all weights are zero
             return random.randint(0, len(operator_list) - 1), operator_list[random.randint(0, len(operator_list) - 1)][2]

        weights = [weight / total_weight for _, weight, _ in operator_list]
        # Handle potential floating point issues leading to sum != 1
        if not math.isclose(sum(weights), 1.0):
             weights = [w / sum(weights) for w in weights]

        try:
            idx = random.choices(range(len(operator_list)), weights=weights, k=1)[0]
        except ValueError as e:
            print(f"Error selecting operator: {e}. Weights: {weights}. Total weight: {total_weight}")
            # Fallback to random selection if weights cause an issue
            idx = random.randint(0, len(operator_list) - 1)


        return idx, operator_list[idx][2]  # Trả về chỉ số và tên toán tử


    def run(self):
        print("Run ALNS")
        start_time = time.time()
        S = self.generate_initial_solution()
        if not S:
            print("  -> ALNS failed to generate an initial solution.")
            return None
        S_star = deepcopy(S)
        O_current = self.evaluate(S)
        O_star = O_current
        print(f"  -> Initial Objective: {O_star:.2f}")
        temperature, gamma = 100, 0.98 # Initial temp and cooling rate
        score_system = { # Scores for different acceptance outcomes
            'new_best': 33,
            'accepted_better': 9,
            'accepted_worse': 13,
            'rejected': 0
        }
        segment_length = max(10, int(self.max_iterations / 10)) # Update weights every 10% of iterations

        # Ghi log hiệu suất toán tử
        log_data = []
        segment_scores = {} # Scores accumulated within a segment

        for i in range(self.max_iterations):
            run_time = time.time() - start_time
            if run_time > self.time_limit:
                print("  -> ALNS time limit reached.")
                break
            print(f"\n--- ALNS Iteration {i} (Temp: {temperature:.2f}) ---")

            # Chọn toán tử hủy
            destroy_idx, destroy_name = self.select_operator(self.destroy_operators)
            destroy_op, _, _ = self.destroy_operators[destroy_idx]
            k = random.randint(2, max(2, int(0.25 * len(self.instance.customers))))
            print(f"  -> Applying {destroy_name} with k={k}...")
            S_partial = destroy_op(S, self.instance, k)

            # Chọn toán tử sửa chữa
            repair_idx, repair_name = self.select_operator(self.repair_operators)
            repair_op, _, _ = self.repair_operators[repair_idx]
            print(f"  -> Applying {repair_name}...")
            # Pass the solver_class to the repair operator if it needs to call MILP
            S_prime = repair_op(S_partial, self.instance, self.solver_class if repair_op == repair_greedy_insert else None)


            if S_prime is None or not S_prime.get('routes'):
                print("  -> Repair failed or resulted in empty routes. Skipping iteration.")
                # Assign a penalty score for failed repair? Or just skip update for these ops.
                log_data.append({
                    'iteration': i,
                    'destroy_operator': destroy_name,
                    'repair_operator': repair_name,
                    'objective': float('inf'),
                    'accepted': False,
                    'best_objective': O_star,
                    'status': 'Repair Failed'
                })
                continue # Skip to next iteration

            # Cải thiện cục bộ (Optimal Charging)
            # Ensure S_prime has a valid route structure before passing to local search
            if S_prime.get('routes') and any(len(route) > 1 for route in S_prime['routes']):
                S_prime_refined = local_search_optimal_charging(S_prime, self.instance, self.name, i)
            else:
                print("  -> S_prime has no valid routes, skipping Optimal Charging Local Search.")
                S_prime_refined = S_prime # Keep the repaired solution as is


            O_prime = self.evaluate(S_prime_refined)
            acceptance_score = score_system['rejected']
            status = 'Rejected'

            if self.accept(O_current, O_prime, temperature):
                S, O_current = deepcopy(S_prime_refined), O_prime
                print(f"  -> Accepted new solution. Current Objective: {O_current:.2f}")
                if O_prime < O_star:
                    S_star, O_star = deepcopy(S_prime_refined), O_prime
                    print(f"  ⭐⭐⭐ New best solution found! Best Objective: {O_star:.2f} ⭐⭐⭐")
                    acceptance_score = score_system['new_best']
                    status = 'New Best'
                else:
                    acceptance_score = score_system['accepted_worse'] # Could be better if O_prime == O_current
                    if O_prime < O_current:
                         acceptance_score = score_system['accepted_better']
                         status = 'Accepted Better'
                    else:
                         status = 'Accepted Worse'
            else:
                print(f"  -> Rejected new solution with objective {O_prime:.2f}")

            # Accumulate scores for the segment
            segment_scores[destroy_name] = segment_scores.get(destroy_name, 0) + acceptance_score
            segment_scores[repair_name] = segment_scores.get(repair_name, 0) + acceptance_score

            # Ghi log cho iteration này
            log_data.append({
                'iteration': i,
                'destroy_operator': destroy_name,
                'repair_operator': repair_name,
                'objective': O_prime,
                'accepted': status,
                'best_objective': O_star,
                'status': status,
                'temperature': temperature,
                'runtime': run_time
            })

            # Update weights at the end of the segment
            if (i + 1) % segment_length == 0 or (i + 1) == self.max_iterations:
                print(f"  -> Updating operator weights after segment.")
                self.update_weights(self.destroy_operators, destroy_idx, acceptance_score, segment_scores)
                self.update_weights(self.repair_operators, repair_idx, acceptance_score, segment_scores)
                segment_scores = {} # Reset segment scores


            temperature *= gamma

        # Lưu log vào file
        log_df = pd.DataFrame(log_data)
        log_dir = r"/content/drive/MyDrive/EVRP-TW-DWC/lp_solver1/ALNS/logs"
        os.makedirs(log_dir, exist_ok=True)
        log_file = os.path.join(log_dir, f"{self.name}_alns_log_{datetime.datetime.now().strftime('%Y%m%d_%H%M%S')}.csv")
        log_df.to_csv(log_file, index=False)
        print(f"  -> ALNS log saved to {log_file}")

        self.best_solution = S_star
        return S_star

In [ ]:
#==============================================================================
# STEP 4: CÁC HÀM RUNNER VÀ SO SÁNH
#==============================================================================

def run_milp_solver(instance_path, time_limit):
    # Define the directory and create it if it doesn't exist
    output_dir = "/content/drive/MyDrive/EVRP-TW-DWC/Untitled Folder"
    os.makedirs(output_dir, exist_ok=True)

    # Generate filename based on current date and time and instance name
    current_time = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
    output_filename = os.path.join(output_dir, f"MILP_{instance_path.stem}_{current_time}.txt")

    original_stdout = sys.stdout # Save original stdout

    try:
        # Redirect stdout to the file
        sys.stdout = open(output_filename, "w")

        print("\n" + "="*60)
        print(f"RUNNING: MILP Solver for {instance_path.name}")
        print("="*60)
        start_run_time = time.time()
        instance = EVRPTWInstance()
        instance.parse_instance_file(str(instance_path))
        instance.set_wireless_coverage("moderate")
        instance.max_vehicles = min(len(instance.customers), 4)
        # Tạo instance
        solver = EnhancedEVRPTWSolver(instance, max_vehicles=instance.max_vehicles)
        solver.create_model()

        # Create directory for MILP logs if it doesn't exist
        milp_log_dir = r"/content/drive/MyDrive/EVRP-TW-DWC/lp_solver1/MILP/logs"
        os.makedirs(milp_log_dir, exist_ok=True)
        milp_log_file = os.path.join(milp_log_dir, f"{instance_path.stem}_milp_log.txt")


        # Bắt đầu giải
        success = solver.solve(time_limit=time_limit, log_file=milp_log_file)
        total_run_time = time.time() - start_run_time

        # Thư mục lưu file LP
        lp_dir = r"/content/drive/MyDrive/EVRP-TW-DWC/lp_solver1/MILP"
        os.makedirs(lp_dir, exist_ok=True)
        lp_filename = os.path.join(lp_dir, f"{instance_path.name}.lp")
        try:
            solver.model.writeLP(lp_filename)
            print(f"📝 Đã ghi mô hình LP ra file: {lp_filename}")
        except Exception as e:
             print(f"Warning: Could not write LP file {lp_filename}: {e}")


        if success and solver.solution:
            print(f"-> MILP finished successfully in {total_run_time:.2f}s.")
            solver.print_solution()
            return {"instance": instance_path.stem, "method": "MILP", "status": "Optimal/Feasible", "vehicles": solver.solution.get('num_vehicles', 0), "total_time": solver.solution.get('total_time', 0), "total_cost": solver.solution.get('total_cost', 0), "solve_time": total_run_time, 'mip_gap': solver.solution.get('mip_gap')} # Include mip_gap
        else:
            print(f"-> MILP failed or found no solution in {total_run_time:.2f}s.")
            return {"instance": instance_path.stem, "method": "MILP", "status": "Failed", "vehicles": "-", "total_time": "-", "total_cost": "-", "solve_time": total_run_time, 'mip_gap': None} # Include mip_gap

    except Exception as e:
        print(f"An error occurred during MILP run for {instance_path.name}: {e}")
        return {"instance": instance_path.stem, "method": "MILP", "status": "Error", "vehicles": "-", "total_time": "-", "total_cost": "-", "solve_time": -1, 'mip_gap': None}

    finally:
        # Restore stdout
        sys.stdout.close()
        sys.stdout = original_stdout
        print(f"Finished MILP run for {instance_path.name}. Output saved to {output_filename}")


def run_advanced_alns(instance_path, time_limit):
    # Define the directory and create it if it doesn't exist
    output_dir = "/content/drive/MyDrive/EVRP-TW-DWC/Untitled Folder"
    os.makedirs(output_dir, exist_ok=True)

    # Generate filename based on current date and time and instance name
    current_time = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
    output_filename = os.path.join(output_dir, f"ALNS_{instance_path.stem}_{current_time}.txt")

    original_stdout = sys.stdout # Save original stdout

    try:
        # Redirect stdout to the file
        sys.stdout = open(output_filename, "w")

        print("\n" + "="*60)
        print(f"RUNNING: Advanced ALNS for {instance_path.name}")
        print("="*60)
        start_run_time = time.time()
        instance = EVRPTWInstance()
        instance.parse_instance_file(str(instance_path))
        instance.set_wireless_coverage("moderate")
        instance.max_vehicles = min(len(instance.customers), 4)
        alns = AdvancedALNS(instance, EnhancedEVRPTWSolver, instance_path.name, max_iterations=100, time_limit=time_limit)
        best_solution = alns.run()
        total_run_time = time.time() - start_run_time

        if best_solution:
            print(f"\n-> Advanced ALNS finished with best solution in {total_run_time:.2f}s.")
            print("\n=== ALNS Final Best Solution ===")
            temp_solver = EnhancedEVRPTWSolver(instance)
            temp_solver.solution = best_solution
            temp_solver.print_solution()
            return {"instance": instance_path.stem, "method": "Advanced ALNS", "status": "Feasible", "vehicles": best_solution.get('num_vehicles', 0), "total_time": best_solution.get('total_time', 0), "total_cost": best_solution.get('total_cost', 0), "solve_time": total_run_time, 'mip_gap': best_solution.get('mip_gap')} # Include mip_gap
        else:
            print(f"-> Advanced ALNS failed to find a solution in {total_run_time:.2f}s.")
            return {"instance": instance_path.stem, "method": "Advanced ALNS", "status": "Failed", "vehicles": "-", "total_time": "-", "total_cost": "-", "solve_time": total_run_time, 'mip_gap': None} # Include mip_gap

    except Exception as e:
        print(f"An error occurred during ALNS run for {instance_path.name}: {e}")
        return {"instance": instance_path.stem, "method": "Advanced ALNS", "status": "Error", "vehicles": "-", "total_time": "-", "total_cost": "-", "solve_time": -1, 'mip_gap': None}

    finally:
        # Restore stdout
        sys.stdout.close()
        sys.stdout = original_stdout
        print(f"Finished Advanced ALNS run for {instance_path.name}. Output saved to {output_filename}")


# Bảng tổng hợp và so sánh các kết quả
def create_comparison_table_advanced(results):
    print("\n\n" + "#"*80)
    print("BẢNG TỔNG HỢP SO SÁNH KẾT QUẢ (MILP vs Advanced ALNS)")
    print("#"*80)
    if not results:
        print("Không có kết quả để hiển thị.")
        return
    df = pd.DataFrame(results)
    df_milp = df[df['method'] == 'MILP'].set_index('instance')
    df_alns = df[df['method'] == 'Advanced ALNS'].set_index('instance')
    df_comp = df_milp.join(df_alns, lsuffix='_milp', rsuffix='_alns')
    def calculate_deviation(val_alns, val_milp):
        if isinstance(val_alns, (int, float)) and isinstance(val_milp, (int, float)) and val_milp > 0:
            return ((val_alns - val_milp) / val_milp) * 100
        return 'N/A'
    df_comp['cost_gap_%'] = df_comp.apply(lambda row: calculate_deviation(row['total_cost_alns'], row['total_cost_milp']), axis=1)
    df_comp['vehicles_gap'] = df_comp.apply(lambda row: row['vehicles_alns'] - row['vehicles_milp'] if isinstance(row['vehicles_alns'], int) and isinstance(row['vehicles_milp'], int) else 'N/A', axis=1)
    # Corrected display_cols to match available columns after join
    display_cols = ['total_cost_milp', 'total_cost_alns', 'cost_gap_%', 'vehicles_milp', 'vehicles_alns', 'vehicles_gap', 'solve_time_milp', 'solve_time_alns', 'mip_gap_milp', 'mip_gap_alns']
    df_display = df_comp[display_cols]
    df_display.columns = ['Cost (MILP)', 'Cost (Adv. ALNS)', 'Cost Gap (%)', 'Vehicles (MILP)', 'Vehicles (Adv. ALNS)', 'Vehicles Gap', 'Time (s) MILP', 'Time (s) ALNS', 'MILP Gap (MILP)', 'MILP Gap (ALNS)'] # Updated column names
    pd.options.display.float_format = '{:,.4f}'.format # Increased precision for costs and gaps
    print(df_display)
#==============================================================================
# STEP 5: KỊCH BẢO CHẠY CHÍNH
#==============================================================================

def main_advanced_comparison():
    instance_dir = Path(r"/content/drive/MyDrive/evrptw_instances")
    instance_list_file = Path("instance_list.txt")
    # test_instances = ["c101C5.txt", "c101C10.txt", "r104C5.txt",  "r202C5.txt", "rc204C5.txt", "rc205C10.txt"]
    test_instances = ["c101C5.txt", "r104C5.txt",  "rc204C5.txt", "rc205C10.txt"]
    with open(instance_list_file, "w") as f:
        for name in test_instances:
            f.write(str(instance_dir / name) + "\n")
    print(f"Đã tạo file '{instance_list_file}' với các instance để chạy.")
    with open(instance_list_file, "r") as f:
        instance_paths = [Path(line.strip()) for line in f if line.strip()]
    TIME_LIMIT_PER_RUN = 3600
    all_results = []
    for path in instance_paths:
        if not path.exists():
            print(f"WARNING: File not found, skipping: {path}")
            continue

        # Chạy MILP
        # milp_result = run_milp_solver(path, time_limit=TIME_LIMIT_PER_RUN)
        # if milp_result: all_results.append(milp_result)

        # Chạy thuật toán ALNS
        alns_result = run_advanced_alns(path, time_limit=TIME_LIMIT_PER_RUN)
        if alns_result: all_results.append(alns_result)

    # Create a final comparison table file
    comparison_output_dir = "/content/drive/MyDrive/EVRP-TW-DWC/Untitled Folder"
    os.makedirs(comparison_output_dir, exist_ok=True)
    comparison_filename = os.path.join(comparison_output_dir, f"comparison_summary_{datetime.datetime.now().strftime('%Y%m%d_%H%M%S')}.txt")

    original_stdout = sys.stdout # Save original stdout
    try:
        sys.stdout = open(comparison_filename, "w")
        create_comparison_table_advanced(all_results)
    except Exception as e:
        print(f"An error occurred while creating the comparison table: {e}")
    finally:
        sys.stdout.close()
        sys.stdout = original_stdout
        print(f"Comparison table saved to: {comparison_filename}")


# --- BẮT ĐẦU CHẠY ---
if __name__ == "__main__":
    main_advanced_comparison()

Đã tạo file 'instance_list.txt' với các instance để chạy.


In [ ]:
# import matplotlib.pyplot as plt

# def plot_alns_routes(alns, instance):
#     """
#     Vẽ các tuyến đường được tạo bởi thuật toán ALNS trên bản đồ.

#     Parameters:
#     - alns: Đối tượng AdvancedALNS chứa giải pháp tốt nhất.
#     - instance: Đối tượng EVRPTWInstance chứa dữ liệu về các nút (nodes).
#     """
#     # Lấy giải pháp tốt nhất từ ALNS
#     best_solution = alns.best_solution
#     if not best_solution or 'routes' not in best_solution:
#         print("Không có giải pháp để vẽ!")
#         return

#     routes = best_solution['routes']
#     nodes = instance.nodes

#     # Tạo từ điển ánh xạ ID nút tới tọa độ
#     node_coords = {node_id: (data['x'], data['y']) for node_id, data in nodes.items()}

#     # Chuẩn bị dữ liệu tọa độ cho các tuyến đường
#     route_coords = []
#     for route in routes:
#         route_sequence = []
#         for node in route:
#             if node in node_coords:
#                 route_sequence.append(node_coords[node])
#         route_coords.append(route_sequence)

#     # Vẽ bản đồ
#     plt.figure(figsize=(12, 12))

#     # Vẽ các tuyến đường
#     for route in route_coords:
#         if route:  # Chỉ vẽ nếu tuyến có ít nhất một điểm
#             x_coords = [point[0] for point in route]
#             y_coords = [point[1] for point in route]
#             plt.plot(x_coords, y_coords, marker='o', linestyle='-', linewidth=2, markersize=8)

#     # Vẽ các điểm đặc biệt: depot, customers, stations
#     depot = node_coords[instance.depot_id]
#     customers = [node_coords[node] for node in instance.customers if node in node_coords]
#     stations = [node_coords[node] for node in instance.stations if node in node_coords]

#     plt.scatter(depot[0], depot[1], c='red', label='Depot', s=150, edgecolor='k', zorder=5)
#     plt.scatter([c[0] for c in customers], [c[1] for c in customers], c='blue', label='Customers', s=100, zorder=5)
#     plt.scatter([s[0] for s in stations], [s[1] for s in stations], c='green', label='Charging Stations', s=100, zorder=5)

#     # Gắn nhãn cho các nút
#     for node_id, coord in node_coords.items():
#         plt.annotate(node_id, (coord[0], coord[1]), textcoords="offset points", xytext=(0, 10), ha='center', fontsize=10)

#     # Tùy chỉnh bản đồ
#     plt.xlabel('Tọa độ X', fontsize=12)
#     plt.ylabel('Tọa độ Y', fontsize=12)
#     plt.title('Tuyến đường EVRP từ thuật toán ALNS', fontsize=14, pad=10)
#     plt.legend(loc='best', fontsize=10)
#     plt.grid(True, linestyle='--', alpha=0.7)
#     plt.tight_layout()

#     # Hiển thị bản đồ
#     plt.show()

# # Cách sử dụng:
# # Sau khi chạy ALNS, gọi hàm này với đối tượng alns và instance


# Task
Visualize the routes generated by the ALNS algorithm on a map.

## Extract route data

### Subtask:
Extract the routes and the coordinates of the nodes from the ALNS solution.


**Reasoning**:
Access the ALNS object to get the best solution and the instance data, then extract the routes and node coordinates as instructed.



# Task
Implement a greedy algorithm to generate an initial solution for the vehicle routing problem and integrate it into the `generate_initial_solution` method of the `AdvancedALNS` class, replacing the current MILP solver call.

## Implement greedy initial solution algorithm

### Subtask:
Implement a greedy algorithm to generate an initial solution for the vehicle routing problem.


**Reasoning**:
Implement the `generate_greedy_solution` function as described in the instructions to create a greedy initial solution for the EVRPTW problem.



# Task
Implement a greedy initial solution algorithm for the EVRP that considers battery levels and integrates it into the `AdvancedALNS` class.

## Implement greedy initial solution algorithm with battery consideration

### Subtask:
Implement a greedy algorithm that constructs initial routes by adding customers based on proximity and considering the current battery level. When the battery is low and the next customer is unreachable, the algorithm should greedily insert a visit to the nearest charging station and charge the battery to full.


**Reasoning**:
Implement the `generate_greedy_solution_battery_aware` function to create a greedy initial solution for the EVRPTW problem, considering battery and time window constraints, and integrating charging station visits when necessary.

